# Hungarian Election Prediction 2026 | Andrási Kristóf - data preparation


### Imports

This block loads the packages used in the notebook.


In [1]:
# this block loads the packages used in the notebook.
import importlib
import subprocess
import sys

# check and install missing packages
def ensure_packages(packages):
    # packages is a list of package names as strings
    for pkg in packages:
        try:
            # try to import the package
            importlib.import_module(pkg)
        except ImportError:
            # if import fails, install with pip
            print(f"{pkg} not found, installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
            print(f"{pkg} installed.")

# first make sure the core packages are available
ensure_packages(["pandas",
                 "numpy",
                 "openpyxl",
                 "matplotlib",
                 "re",
                 "xlrd"])

# now import the packages you actually want to use
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import re
import numpy as np

### Working Directory

This block sets the main project paths so file loading works the same way each time.


In [2]:
# this block sets the main project paths so file loading works the same way each time.
def find_project_root(start_path=None):
    start_path = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "scripts" / "final_final_scripts").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root from the current working directory.")

here = Path.cwd().resolve()
project_root = find_project_root(here)
data_dir = project_root / "data" / "TokaGabor"

print("here:", here)
print("project_root:", project_root)
print("data_dir:", data_dir)


here: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions/scripts/final_final_scripts
project_root: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions
data_dir: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions/data/TokaGabor


### Polls

This block loads and checks the poll data used later in the project.


In [3]:
# this block loads and checks the poll data used later in the project.

polls_path = data_dir / "Nyers_adatok" / "DATA_Választási_közvélemény-kutatások_Magyarországon.xlsx"

polls = pd.read_excel(
    polls_path,
    sheet_name="Adatok"
)

# drop polls with a recorded sample size of zero at the source
poll_sample_size = pd.to_numeric(polls["Minta"], errors="coerce")
polls = polls.loc[poll_sample_size.ne(0) | poll_sample_size.isna()].copy()
del poll_sample_size

del(polls_path)

This block checks the available column names in the table.


In [4]:
# this block checks the available column names in the table.
polls.columns

Index(['Adatok.bázisa', 'Kezdet', 'Vég', 'Adatgazda', 'Kalibrált', 'Fókusz',
       'Mód', 'Minta', 'Fidesz', 'MSZP', 'Jobbik', 'LMP', 'DK', 'Együtt', 'P',
       'MM', 'DK-MSZP-P', 'MSZP-P', 'MMN', 'NP', 'EM', 'MKKP', '2RK', 'MH',
       'TISZA', 'Egyéb párt', 'Egyéb válasz', 'alpha', 'Summa',
       'Biztos szavazó', 'FFALU', 'O1GBP', 'Megrendelő', 'Forrás 1',
       'Forrás 2', 'Forrás 3', 'Megjegyzések', 'UnixStart', 'UnixEnd',
       'UnixTime'],
      dtype='object')

### Previous years national level results

#### national results

This block builds the national election result table used later in the notebook.


In [5]:
# this block builds the national election result table used later in the notebook.
import pandas as pd

# 2018 parliamentary mandates by party
mandates_2018 = pd.DataFrame({
    "party": [
        "Fidesz–KDNP",
        "Jobbik",
        "MSZP–Párbeszéd",
        "DK",
        "LMP",
        "MNOÖ",
        "Független",
        "EGYÜTT",
        "Összesen"
    ],
    "single_seats": [91, 1, 8, 3, 1, 0, 1, 1, 106],
    "list_seats":   [42, 25, 12, 6, 7, 1, 0, 0, 93],
    "total_seats":  [133, 26, 20, 9, 8, 1, 1, 1, 199],
    "percent":      [66.83, 13.07, 10.05, 4.52, 4.03, 0.50, 0.50, 0.50, 100.00]
})

# 2022 parliamentary mandates by party
mandates_2022 = pd.DataFrame({
    "party": [
        "Fidesz–KDNP",
        "DK–Jobbik–Momentum–MSZP–LMP–Párbeszéd",
        "Mi Hazánk",
        "MNOÖ",
        "Összesen"
    ],
    "single_seats": [87, 19, 0, 0, 106],
    "list_seats":   [48, 38, 6, 1, 93],
    "total_seats":  [135, 57, 6, 1, 199],
    "percent":      [67.84, 28.64, 3.02, 0.50, 100.00]
})

# add year
mandates_2018["year"] = 2018
mandates_2022["year"] = 2022

# 2018: keep only oevk-level parties + others
keep_2018 = [
    "Fidesz–KDNP",
    "Jobbik",
    "MSZP–Párbeszéd",
    "DK",
    "LMP",
    "EGYÜTT"
]

mandates_2018_main = mandates_2018[
    mandates_2018["party"].isin(keep_2018)
].copy()

mandates_2018_other_src = mandates_2018[
    ~mandates_2018["party"].isin(keep_2018 + ["Összesen"])
].copy()

others_2018 = pd.DataFrame([{
    "year": 2018,
    "party": "Others",
    "single_seats": mandates_2018_other_src["single_seats"].sum(),
    "list_seats":   mandates_2018_other_src["list_seats"].sum(),
    "total_seats":  mandates_2018_other_src["total_seats"].sum(),
    "percent":      mandates_2018_other_src["percent"].sum()
}])

mandates_2018_clean = pd.concat(
    [mandates_2018_main, others_2018],
    ignore_index=True
)

# rename parties to match telep/oevk party_name for 2018
name_map_2018 = {
    "DK": "Demokratikus Koalíció",
    "LMP": "Lehet Más a Politika",
    "EGYÜTT": "Együtt"
}
mandates_2018_clean["party"] = mandates_2018_clean["party"].replace(name_map_2018)

# add momentum and mkkp with zero seats for 2018
extra_2018 = pd.DataFrame({
    "year": [2018, 2018],
    "party": ["Momentum", "MKKP"],
    "single_seats": [0, 0],
    "list_seats": [0, 0],
    "total_seats": [0, 0],
    "percent": [0.0, 0.0]
})

mandates_2018_clean = pd.concat(
    [mandates_2018_clean, extra_2018],
    ignore_index=True
)

# 2022: keep only oevk-level parties + others
keep_2022 = [
    "Fidesz–KDNP",
    "DK–Jobbik–Momentum–MSZP–LMP–Párbeszéd",
    "Mi Hazánk"
]

mandates_2022_main = mandates_2022[
    mandates_2022["party"].isin(keep_2022)
].copy()

mandates_2022_other_src = mandates_2022[
    ~mandates_2022["party"].isin(keep_2022 + ["Összesen"])
].copy()

others_2022 = pd.DataFrame([{
    "year": 2022,
    "party": "Others",
    "single_seats": mandates_2022_other_src["single_seats"].sum(),
    "list_seats":   mandates_2022_other_src["list_seats"].sum(),
    "total_seats":  mandates_2022_other_src["total_seats"].sum(),
    "percent":      mandates_2022_other_src["percent"].sum()
}])

mandates_2022_clean = pd.concat(
    [mandates_2022_main, others_2022],
    ignore_index=True
)

# party_block for both years
party_block_2018 = {
    "Fidesz–KDNP": "gov",
    "Jobbik": "opp_radical",
    "MSZP–Párbeszéd": "opp",
    "Demokratikus Koalíció": "opp",
    "Lehet Más a Politika": "opp",
    "Együtt": "other",
    "Momentum": "opp",
    "MKKP": "other",
    "Others": "other"
}

party_block_2022 = {
    "Fidesz–KDNP": "gov",
    "DK–Jobbik–Momentum–MSZP–LMP–Párbeszéd": "opp",
    "Mi Hazánk": "opp_radical",
    "Others": "other"
}

mandates_2018_clean["party_block"] = mandates_2018_clean["party"].map(party_block_2018)
mandates_2022_clean["party_block"] = mandates_2022_clean["party"].map(party_block_2022)

# enforce column order
mandates_2018_clean = mandates_2018_clean[
    ["year", "party", "party_block", "single_seats", "list_seats", "total_seats", "percent"]
]
mandates_2022_clean = mandates_2022_clean[
    ["year", "party", "party_block", "single_seats", "list_seats", "total_seats", "percent"]
]

# combine 2018 and 2022
mandates_all = pd.concat(
    [mandates_2018_clean, mandates_2022_clean],
    ignore_index=True
)

# clean up everything except mandates_all
for var in [
    "mandates_2018", "mandates_2022",
    "mandates_2018_main", "mandates_2018_other_src", "others_2018",
    "extra_2018", "mandates_2018_clean",
    "mandates_2022_main", "mandates_2022_other_src", "others_2022",
    "mandates_2022_clean",
    "party_block_2018", "party_block_2022",
    "keep_2018", "keep_2022",
    "name_map_2018"
]:
    if var in locals():
        del locals()[var]
del var


#### ep results

Source: https://vtr.valasztas.hu/ep2024 ; https://valtor.valasztas.hu/valtort/jsp/orszjkv.jsp?EA=39&W=11.


In [6]:
# source: https://vtr.valasztas.hu/ep2024 ; https://valtor.valasztas.hu/valtort/jsp/orszjkv.jsp?EA=39&W=11


# 2019 ep mandates and votes (national level)
ep_2019 = pd.DataFrame({
    "party": [
        "Fidesz–KDNP",
        "Jobbik",
        "Momentum",
        "Mi Hazánk",
        "MSZP–Párbeszéd",
        "Lehet Más a Politika",
        "Demokratikus Koalíció",
        "Munkáspárt",
        "MKKP",
        "Others"
    ],
    # valid list votes (national)
    "list_votes": [
        1824220,   # Fidesz–KDNP
        220184,    # Jobbik
        344512,    # Momentum
        114156,    # Mi Hazánk
        229551,    # MSZP–Párbeszéd
        75498,     # LMP
        557081,    # DK
        14452,     # Munkáspárt
        90912,     # MKKP
        0          # Others (residual here taken as 0)
    ],
    # vote share in percent
    "percent": [
        52.56,  # Fidesz–KDNP
        6.34,   # Jobbik
        9.93,   # Momentum
        3.29,   # Mi Hazánk
        6.61,   # MSZP–Párbeszéd
        2.18,   # LMP
        16.05,  # DK
        0.42,   # Munkáspárt
        2.62,   # MKKP
        0.0     # Others
    ],
    # ep mandates
    "seats": [
        13,  # Fidesz–KDNP
        1,   # Jobbik
        2,   # Momentum
        0,   # Mi Hazánk
        1,   # MSZP–Párbeszéd
        0,   # LMP
        4,   # DK
        0,   # Munkáspárt
        0,   # MKKP
        0    # Others
    ]
})

ep_2019["year"] = 2019

# 2024 ep mandates and votes (national level)
ep_2024 = pd.DataFrame({
    "party": [
        "Fidesz–KDNP",
        "TISZA",
        "DK–MSZP–P",
        "Mi Hazánk",
        "Momentum",
        "Kutyapárt",
        "Jobbik",
        "LMP",
        "2RK",
        "Mindenki Magyarországa",
        "Megoldás",
        "Others"
    ],
    "list_votes": [
        2048211,  # Fidesz–KDNP
        1352699,  # TISZA
        367162,   # DK–MSZP–P
        306404,   # Mi Hazánk
        169082,   # Momentum
        163960,   # Kutyapárt (MKKP)
        45404,    # Jobbik
        39646,    # LMP
        30961,    # 2RK
        29285,    # Mindenki Magyarországa (MMN)
        16806,    # Megoldás (MEMO)
        0         # Others (residual here taken as 0)
    ],
    "percent": [
        44.82,  # Fidesz–KDNP
        29.60,  # TISZA
        8.03,   # DK–MSZP–P
        6.71,   # Mi Hazánk
        3.70,   # Momentum
        3.59,   # Kutyapárt
        0.99,   # Jobbik
        0.87,   # LMP
        0.68,   # 2RK
        0.64,   # Mindenki Magyarországa
        0.37,   # Megoldás
        0.0     # Others
    ],
    "seats": [
        11,  # Fidesz–KDNP
        7,   # TISZA
        2,   # DK–MSZP–P
        1,   # Mi Hazánk
        0,   # Momentum
        0,   # Kutyapárt
        0,   # Jobbik
        0,   # LMP
        0,   # 2RK
        0,   # Mindenki Magyarországa
        0,   # Megoldás
        0    # Others
    ]
})

ep_2024["year"] = 2024

# add party_block consistent with telep/oevk definitions

party_block_2019 = {
    "Fidesz–KDNP": "gov",
    "Jobbik": "opp",
    "Momentum": "opp",
    "Mi Hazánk": "opp_radical",
    "MSZP–Párbeszéd": "opp",
    "Lehet Más a Politika": "opp",
    "Demokratikus Koalíció": "opp",
    "Munkáspárt": "other",
    "MKKP": "other",
    "Others": "other"
}

party_block_2024 = {
    "Fidesz–KDNP": "gov",
    "Mi Hazánk": "opp_radical",
    "TISZA": "opp",
    "DK–MSZP–P": "opp",
    "LMP": "other",
    "Kutyapárt": "other",
    "Megoldás": "other",
    "2RK": "other",
    "Momentum": "other",
    "Jobbik": "other",
    "Mindenki Magyarországa": "other",
    "Others": "other"
}

ep_2019["party_block"] = ep_2019["party"].map(party_block_2019)
ep_2024["party_block"] = ep_2024["party"].map(party_block_2024)

# enforce final column order
ep_2019 = ep_2019[
    ["year", "party", "party_block", "list_votes", "percent", "seats"]
]
ep_2024 = ep_2024[
    ["year", "party", "party_block", "list_votes", "percent", "seats"]
]

# combine into one table
ep_mandates_all = pd.concat(
    [ep_2019, ep_2024],
    ignore_index=True
)

# clean up everything except ep_mandates_all
for var in [
    "ep_2019",
    "ep_2024",
    "party_block_2019",
    "party_block_2024"
]:
    if var in locals():
        del locals()[var]
del var


#### merge

Work on copies so originals stay intact.


In [7]:
# work on copies so originals stay intact
nat = mandates_all.copy()
ep = ep_mandates_all.copy()

# add election_type
nat["election_type"] = "nat"
ep["election_type"] = "ep"

# make columns compatible

# national table has single/list/total seats, but no list_votes or seats column
nat["list_votes"] = pd.NA              # no national list vote total in this table
nat["seats"] = nat["total_seats"]      # for compatibility with ep_mandates_all

# ep table has list_votes + seats, but no single/list/total_seats (parliament style)
ep["single_seats"] = pd.NA
ep["list_seats"] = pd.NA
ep["total_seats"] = ep["seats"]

# define a common column order
common_cols = [
    "year",
    "election_type",
    "party",
    "party_block",
    "single_seats",
    "list_seats",
    "total_seats",
    "list_votes",
    "percent",
    "seats"
]

# concatenate into one combined mandates table
mandates_all_combined = pd.concat(
    [
        nat[common_cols],
        ep[common_cols]
    ],
    ignore_index=True
)

# optionally, if you don't need both total_seats and seats,
# you can drop one of them, pl.:
# mandates_all_combined = mandates_all_combined.drop(columns=["total_seats"])

# clean up temporary variables
for var in ["nat", "ep", "common_cols"]:
    if var in locals():
        del locals()[var]
del var, ep_mandates_all, mandates_all

### Previous years OEVK level results
#### 2018

Path to 2018 oevk excel.


In [8]:
# path to 2018 oevk excel
oevk_path_2018 = data_dir / "Választási_eredmények" / "2018.04.08.végleges_eredmények" / "oevk_szerinti_eredmenyek_2018.xlsx"

# read raw excel
raw_2018 = pd.read_excel(oevk_path_2018, sheet_name="Adatok")

# rename key id and total columns
raw_2018 = raw_2018.rename(columns={
    "Megyekód": "county_code",
    "Megyenév": "county_name",
    "Megye": "county_name",
    "OEVK": "oevk_code",
    # candidate level totals
    "AE": "electorate_cand",
    "JE": "voters_cand",
    "NE": "valid_cand_total",
    # list level totals
    "AL": "electorate_list",
    "JL": "voters_list",
    "NL": "valid_list_total"
})

# add year column
raw_2018["year"] = 2018

# setup main parties:
# cand_col = column with candidate votes
# list_col = column with list votes
# party_block = gov / opp / opp_radical
main_parties = [
    {
        "party_id": "FIDESZ_KDNP",
        "party_name": "Fidesz–KDNP",
        "party_block": "gov",
        "cand_col": "FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG, KERESZTÉNYDEMOKRATA NÉPPÁRT",
        "list_col": "FIDESZ-KDNP"
    },
    {
        "party_id": "JOBBIK",
        "party_name": "Jobbik",
        "party_block": "opp_radical",
        "cand_col": "JOBBIK MAGYARORSZÁGÉRT MOZGALOM",
        "list_col": "JOBBIK"
    },
    {
        "party_id": "MSZP_PARBESZED",
        "party_name": "MSZP–Párbeszéd",
        "party_block": "opp",
        "cand_col": "MAGYAR SZOCIALISTA PÁRT, PÁRBESZÉD MAGYARORSZÁGÉRT PÁRT",
        "list_col": "MSZP-PÁRBESZÉD"
    },
    {
        "party_id": "DK",
        "party_name": "Demokratikus Koalíció",
        "party_block": "opp",
        "cand_col": "DEMOKRATIKUS KOALÍCIÓ",
        "list_col": "DK"
    },
    {
        "party_id": "LMP",
        "party_name": "Lehet Más a Politika",
        "party_block": "opp",
        "cand_col": "LEHET MÁS A POLITIKA",
        "list_col": "LMP"
    },
    {
        "party_id": "EGYUTT",
        "party_name": "Együtt",
        "party_block": "opp",
        "cand_col": "EGYÜTT - A KORSZAKVÁLTÓK PÁRTJA",
        "list_col": "EGYÜTT"
    },
    {
        "party_id": "MOMENTUM",
        "party_name": "Momentum",
        "party_block": "opp",
        "cand_col": "MOMENTUM MOZGALOM",
        "list_col": "MOMENTUM"
    },
    {
        "party_id": "MKKP",
        "party_name": "MKKP",
        "party_block": "other",
        "cand_col": "MAGYAR KÉTFARKÚ KUTYA PÁRT",
        "list_col": "MKKP"
    }
]

# base columns that are repeated for each party
base_cols = [
    "year",
    "county_code",
    "county_name",
    "oevk_code",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total"
]

rows = []

# build long-form rows for each main party
for cfg in main_parties:
    cand_col = cfg["cand_col"]
    list_col = cfg["list_col"]
    
    # start from base columns
    tmp = raw_2018[base_cols].copy()
    
    # add party metadata
    tmp["party_id"] = cfg["party_id"]
    tmp["party_name"] = cfg["party_name"]
    tmp["party_block"] = cfg["party_block"]
    
    # candidate and list votes (if column missing, use 0)
    tmp["cand_votes"] = raw_2018.get(cand_col, 0)
    tmp["list_votes"] = raw_2018.get(list_col, 0)
    
    # vote shares, with protection against division by zero
    tmp["cand_share"] = tmp["cand_votes"] / tmp["valid_cand_total"].replace({0: pd.NA})
    tmp["list_share"] = tmp["list_votes"] / tmp["valid_list_total"].replace({0: pd.NA})
    
    rows.append(tmp)

# combine all main parties into one long table
oevk_party_results_2018 = pd.concat(rows, ignore_index=True)

# now add an "others" row per oevk using residual votes
# first compute sums of main party votes at oevk level
cand_cols_main = [p["cand_col"] for p in main_parties]
list_cols_main = [p["list_col"] for p in main_parties]

cand_main_sum = raw_2018[cand_cols_main].sum(axis=1)
list_main_sum = raw_2018[list_cols_main].sum(axis=1)

others_cand_votes = (raw_2018["valid_cand_total"] - cand_main_sum).clip(lower=0)
others_list_votes = (raw_2018["valid_list_total"] - list_main_sum).clip(lower=0)

others = raw_2018[base_cols].copy()
others["party_id"] = "OTHERS"
others["party_name"] = "Others"
others["party_block"] = "other"
others["cand_votes"] = others_cand_votes
others["list_votes"] = others_list_votes
others["cand_share"] = others["cand_votes"] / others["valid_cand_total"].replace({0: pd.NA})
others["list_share"] = others["list_votes"] / others["valid_list_total"].replace({0: pd.NA})

# add others to the long table
oevk_party_results_2018 = pd.concat(
    [oevk_party_results_2018, others],
    ignore_index=True
)


# clean up everything except the final table
del (
    oevk_path_2018,
    raw_2018,
    main_parties,
    base_cols,
    rows,
    cfg,
    cand_col,
    list_col,
    tmp,
    cand_cols_main,
    list_cols_main,
    cand_main_sum,
    list_main_sum,
    others_cand_votes,
    others_list_votes,
    others
)


#### 2022

Path to 2022 oevk excel.


In [9]:
# path to 2022 oevk excel
oevk_path_2022 = data_dir / "Választási_eredmények" / "2022.04.03._végleges_eredmények" / "2022.04.03._valasztasi_eredmenyek_Vox_Populi_adatbazis.xlsx"

# read raw excel (oevk-level)
raw_2022 = pd.read_excel(oevk_path_2022, sheet_name="Választókerületi adatok")

# rename key id and total columns
raw_2022 = raw_2022.rename(columns={
    "OEVKoffice": "oevk_office",
    "OEVKname": "oevk_name",
    "maz": "county_code",
    "evk": "oevk_code",
    # candidate-level totals
    "E.a": "electorate_cand",
    "E.j": "voters_cand",
    "E.n": "valid_cand_total",
    # list-level totals
    "L.a": "electorate_list",
    "L.j": "voters_list",
    "L.n": "valid_list_total"
})

# add year column
raw_2022["year"] = 2022

# setup main parties:
# cand_col = column with candidate votes
# list_col = column with list votes
# party_block = gov / opp / opp_radical
main_parties_2022 = [
    {
        "party_id": "FIDESZ_KDNP",
        "party_name": "Fidesz–KDNP",
        "party_block": "gov",
        "cand_col": "E.FK",
        "list_col": "L.FK"
    },
    {
        "party_id": "UNITED_OPP",
        "party_name": "DK–Jobbik–Momentum–MSZP–LMP–Párbeszéd",
        "party_block": "opp",
        "cand_col": "E.EM",
        "list_col": "L.EM"
    },
    {
        "party_id": "MI_HAZANK",
        "party_name": "Mi Hazánk",
        "party_block": "opp_radical",
        "cand_col": "E.MH",
        "list_col": "L.MH"
    }
]

# base columns that are repeated for each party
base_cols_2022 = [
    "year",
    "county_code",
    "oevk_code",
    "oevk_name",
    "oevk_office",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total"
]

rows_2022 = []

# build long-form rows for each main party
for cfg in main_parties_2022:
    cand_col = cfg["cand_col"]
    list_col = cfg["list_col"]
    
    # start from base columns
    tmp = raw_2022[base_cols_2022].copy()
    
    # add party metadata
    tmp["party_id"] = cfg["party_id"]
    tmp["party_name"] = cfg["party_name"]
    tmp["party_block"] = cfg["party_block"]
    
    # candidate and list votes (if column missing, use 0)
    tmp["cand_votes"] = raw_2022.get(cand_col, 0)
    tmp["list_votes"] = raw_2022.get(list_col, 0)
    
    # vote shares, with protection against division by zero
    tmp["cand_share"] = tmp["cand_votes"] / tmp["valid_cand_total"].replace({0: pd.NA})
    tmp["list_share"] = tmp["list_votes"] / tmp["valid_list_total"].replace({0: pd.NA})
    
    rows_2022.append(tmp)

# combine all main parties into one long table
oevk_party_results_2022 = pd.concat(rows_2022, ignore_index=True)

# now add an "others" row per oevk using residual votes
# first compute sums of main party votes at oevk level
cand_cols_main_2022 = [p["cand_col"] for p in main_parties_2022]
list_cols_main_2022 = [p["list_col"] for p in main_parties_2022]

cand_main_sum_2022 = raw_2022[cand_cols_main_2022].sum(axis=1)
list_main_sum_2022 = raw_2022[list_cols_main_2022].sum(axis=1)

others_cand_votes_2022 = (raw_2022["valid_cand_total"] - cand_main_sum_2022).clip(lower=0)
others_list_votes_2022 = (raw_2022["valid_list_total"] - list_main_sum_2022).clip(lower=0)

others_2022 = raw_2022[base_cols_2022].copy()
others_2022["party_id"] = "OTHERS"
others_2022["party_name"] = "Others"
others_2022["party_block"] = "other"
others_2022["cand_votes"] = others_cand_votes_2022
others_2022["list_votes"] = others_list_votes_2022
others_2022["cand_share"] = others_2022["cand_votes"] / others_2022["valid_cand_total"].replace({0: pd.NA})
others_2022["list_share"] = others_2022["list_votes"] / others_2022["valid_list_total"].replace({0: pd.NA})

# add others to the long table
oevk_party_results_2022 = pd.concat(
    [oevk_party_results_2022, others_2022],
    ignore_index=True
)


# clean up everything from this cell except the final table
del (
    oevk_path_2022,
    raw_2022,
    main_parties_2022,
    base_cols_2022,
    rows_2022,
    cfg,
    cand_col,
    list_col,
    tmp,
    cand_cols_main_2022,
    list_cols_main_2022,
    cand_main_sum_2022,
    list_main_sum_2022,
    others_cand_votes_2022,
    others_list_votes_2022,
    others_2022
)

#### clean

Merging the two.


In [10]:
# merging the two

# list all columns that appear in either table
cols_2018 = set(oevk_party_results_2018.columns)
cols_2022 = set(oevk_party_results_2022.columns)
all_cols = cols_2018 | cols_2022

# add missing columns with na so the schemas match
for col in all_cols:
    if col not in cols_2018:
        oevk_party_results_2018[col] = pd.NA
    if col not in cols_2022:
        oevk_party_results_2022[col] = pd.NA

# optional: put columns in the same order (use 2018 order + any extra)
ordered_cols = list(oevk_party_results_2018.columns)
# just in case there are cols present only in 2022
for col in all_cols:
    if col not in ordered_cols:
        ordered_cols.append(col)

oevk_party_results_2018 = oevk_party_results_2018[ordered_cols]
oevk_party_results_2022 = oevk_party_results_2022[ordered_cols]

# now safe to stack them
oevk_party_results_all = pd.concat(
    [oevk_party_results_2018, oevk_party_results_2022],
    ignore_index=True
)

del cols_2018, cols_2022, all_cols, ordered_cols, oevk_party_results_2018, oevk_party_results_2022

This block prepares the next part of the notebook.


In [11]:
# this block prepares the next part of the notebook.
del(col)

Make a unique oevk id from county_code + oevk_code.


In [12]:
# make a unique oevk id from county_code + oevk_code
oevk_party_results_all["oevk_id"] = (
    oevk_party_results_all["county_code"].astype(str)
    + "_"
    + oevk_party_results_all["oevk_code"].astype(str)
)

# base: one row per year × oevk_id
base = (
    oevk_party_results_all
    .groupby(["year", "oevk_id"], as_index=False)
    .agg(
        county_code=("county_code", "first"),
        county_name=("county_name", "first"),
        oevk_code=("oevk_code", "first"),
        oevk_name=("oevk_name", "first"),
        oevk_office=("oevk_office", "first"),
        electorate_cand=("electorate_cand", "first"),
        voters_cand=("voters_cand", "first"),
        valid_cand_total=("valid_cand_total", "first"),
        electorate_list=("electorate_list", "first"),
        voters_list=("voters_list", "first"),
        valid_list_total=("valid_list_total", "first")
    )
)

print("rows in base (year × oevk_id):", len(base))

rows in base (year × oevk_id): 212


Fill the missing values from 2018 based on 22 office names.


In [13]:
# fill the missing values from 2018 based on 22 office names
lookup_2022 = (
    oevk_party_results_all[oevk_party_results_all["year"] == 2022]
    .drop_duplicates(subset="oevk_id")
    .set_index("oevk_id")[["oevk_name", "oevk_office"]]
)

# mask for 2018 rows
mask_2018 = oevk_party_results_all["year"] == 2018

# fill 2018 oevk_name from 2022 values
oevk_party_results_all.loc[mask_2018, "oevk_name"] = (
    oevk_party_results_all.loc[mask_2018, "oevk_id"].map(lookup_2022["oevk_name"])
)

# fill 2018 oevk_office from 2022 values
oevk_party_results_all.loc[mask_2018, "oevk_office"] = (
    oevk_party_results_all.loc[mask_2018, "oevk_id"].map(lookup_2022["oevk_office"])
)

# clean up temp objects
for var in ["lookup_2022", "mask_2018"]:
    if var in locals():
        del locals()[var]

del(var)        


Define blocks.


In [14]:
# define blocks
blocks = ["gov", "opp", "opp_radical", "other"]

# sum votes by block within each year × oevk_id
block_votes = (
    oevk_party_results_all
    .groupby(["year", "oevk_id", "party_block"], as_index=False)
    .agg(
        cand_votes=("cand_votes", "sum"),
        list_votes=("list_votes", "sum")
    )
)

# pivot blocks to wide format
block_wide = block_votes.pivot(
    index=["year", "oevk_id"],
    columns="party_block",
    values=["cand_votes", "list_votes"]
)

block_wide = block_wide.fillna(0)

# flatten multiindex columns: cand_votes_gov, list_votes_opp, stb.
block_wide.columns = [f"{kind}_{blk}" for kind, blk in block_wide.columns]

# make sure every block column exists
for blk in blocks:
    for kind in ["cand_votes", "list_votes"]:
        col = f"{kind}_{blk}"
        if col not in block_wide.columns:
            block_wide[col] = 0

block_wide = block_wide.reset_index()

# merge base info with block votes
oevk_block_panel = base.merge(
    block_wide,
    on=["year", "oevk_id"],
    how="left"
)

print("rows in merged panel:", len(oevk_block_panel))

# turnout
oevk_block_panel["turnout_cand"] = (
    oevk_block_panel["voters_cand"] /
    oevk_block_panel["electorate_cand"].where(oevk_block_panel["electorate_cand"] != 0)
)

oevk_block_panel["turnout_list"] = (
    oevk_block_panel["voters_list"] /
    oevk_block_panel["electorate_list"].where(oevk_block_panel["electorate_list"] != 0)
)

# sum of block votes for shares
oevk_block_panel["cand_votes_sum_blocks"] = (
    oevk_block_panel[[f"cand_votes_{b}" for b in blocks]].sum(axis=1)
)

oevk_block_panel["list_votes_sum_blocks"] = (
    oevk_block_panel[[f"list_votes_{b}" for b in blocks]].sum(axis=1)
)

for blk in blocks:
    oevk_block_panel[f"{blk}_cand_share"] = (
        oevk_block_panel[f"cand_votes_{blk}"] /
        oevk_block_panel["cand_votes_sum_blocks"].where(oevk_block_panel["cand_votes_sum_blocks"] != 0)
    )
    oevk_block_panel[f"{blk}_list_share"] = (
        oevk_block_panel[f"list_votes_{blk}"] /
        oevk_block_panel["list_votes_sum_blocks"].where(oevk_block_panel["list_votes_sum_blocks"] != 0)
    )

# quick check: shares sum to ~1
oevk_block_panel["cand_share_sum"] = (
    oevk_block_panel[[f"{b}_cand_share" for b in blocks]].sum(axis=1)
)
oevk_block_panel["list_share_sum"] = (
    oevk_block_panel[[f"{b}_list_share" for b in blocks]].sum(axis=1)
)

print(oevk_block_panel["cand_share_sum"].describe().round(3))
print(oevk_block_panel["list_share_sum"].describe().round(3))

# keep only modelling columns
keep_cols = [
    "year",
    "oevk_id",
    "county_code",
    "county_name",
    "oevk_code",
    "oevk_name",
    "oevk_office",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total",
    "turnout_cand",
    "turnout_list",
    "gov_cand_share",
    "opp_cand_share",
    "opp_radical_cand_share",
    "other_cand_share",
    "gov_list_share",
    "opp_list_share",
    "opp_radical_list_share",
    "other_list_share"
]

# oevk_block_panel = oevk_block_panel[keep_cols]

print("final panel shape:", oevk_block_panel.shape)

# clean up helpers
for var in ["base", "block_votes", "block_wide", "keep_cols", "blocks"]:
    if var in locals():
        del locals()[var]

rows in merged panel: 212
count    212.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: cand_share_sum, dtype: float64
count    212.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: list_share_sum, dtype: float64
final panel shape: (212, 35)


Quickly fill the 2018 NAs from 2022.


In [15]:
# quickly fill the 2018 NAs from 2022

# build lookup table from 2022 oevk data
lookup_oevk = (
    oevk_party_results_all[oevk_party_results_all["year"] == 2022]
    .groupby("oevk_id", as_index=False)[["oevk_name", "oevk_office"]]
    .agg({"oevk_name": "first", "oevk_office": "first"})
)

# make simple dicts for mapping
name_map = dict(zip(lookup_oevk["oevk_id"], lookup_oevk["oevk_name"]))
office_map = dict(zip(lookup_oevk["oevk_id"], lookup_oevk["oevk_office"]))

# make sure columns exist in panel
for col in ["oevk_name", "oevk_office"]:
    if col not in oevk_block_panel.columns:
        oevk_block_panel[col] = pd.NA

# fill only 2018 rows using the 2022 mapping
mask_2018 = oevk_block_panel["year"] == 2018

oevk_block_panel.loc[mask_2018, "oevk_name"] = (
    oevk_block_panel.loc[mask_2018, "oevk_id"].map(name_map)
)

oevk_block_panel.loc[mask_2018, "oevk_office"] = (
    oevk_block_panel.loc[mask_2018, "oevk_id"].map(office_map)
)

# optional: cleanup small helper objects
del lookup_oevk, name_map, office_map, mask_2018

This block prepares the next part of the notebook.


In [16]:
# this block prepares the next part of the notebook.
del(blk, kind, var)

This block prepares the next part of the notebook.


In [17]:
# this block prepares the next part of the notebook.
del(col)

This block checks the available column names in the table.


In [18]:
# this block checks the available column names in the table.
oevk_party_results_all.columns

Index(['year', 'county_code', 'county_name', 'oevk_code', 'electorate_cand',
       'voters_cand', 'valid_cand_total', 'electorate_list', 'voters_list',
       'valid_list_total', 'party_id', 'party_name', 'party_block',
       'cand_votes', 'list_votes', 'cand_share', 'list_share', 'oevk_name',
       'oevk_office', 'oevk_id'],
      dtype='object')

### Previous years settlement level results
Handling the OEVK change in 2026. This is a difficult problem because I have to calculate everything at settlement level, not OEVK level, and then reassign the OEVKs to the settlements.

In [19]:
# this block checks the available column names in the table.
oevk_block_panel.columns

Index(['year', 'oevk_id', 'county_code', 'county_name', 'oevk_code',
       'oevk_name', 'oevk_office', 'electorate_cand', 'voters_cand',
       'valid_cand_total', 'electorate_list', 'voters_list',
       'valid_list_total', 'cand_votes_gov', 'cand_votes_opp',
       'cand_votes_opp_radical', 'cand_votes_other', 'list_votes_gov',
       'list_votes_opp', 'list_votes_opp_radical', 'list_votes_other',
       'turnout_cand', 'turnout_list', 'cand_votes_sum_blocks',
       'list_votes_sum_blocks', 'gov_cand_share', 'gov_list_share',
       'opp_cand_share', 'opp_list_share', 'opp_radical_cand_share',
       'opp_radical_list_share', 'other_cand_share', 'other_list_share',
       'cand_share_sum', 'list_share_sum'],
      dtype='object')

#### 2018

Path to 2018 settlement-level excel.


In [20]:
# path to 2018 settlement-level excel
telep_path_2018 = data_dir / "Választási_eredmények" / "2018.04.08.végleges_eredmények" / "telepulesi_eredmenyek_2018.xlsx"

# read raw excel
raw_2018 = pd.read_excel(telep_path_2018, sheet_name="Adatok")

# rename id and total columns to simpler names
raw_2018 = raw_2018.rename(columns={
    "Megyekód": "county_code",
    "Településkód": "settlement_code",
    "Település": "settlement_name",
    # candidate level totals (egyéni)
    "AE": "electorate_cand",   # névjegyzékben lévő választópolgárok száma
    "JE": "voters_cand",       # szavazó választópolgárok összesen
    "NE": "valid_cand_total",  # érvényes szavazólapok száma
    # list level totals (pártlista)
    "AL": "electorate_list",   # névjegyzékben lévő választópolgárok száma
    "JL": "voters_list",       # szavazó választópolgárok összesen
    "NL": "valid_list_total"   # érvényes listás szavazatok (összesen)
})

# add year column
raw_2018["year"] = 2018

# setup main parties:
# cand_col = column with candidate votes
# list_col = column with list votes
# party_block = gov / opp / opp_radical
main_parties = [
    {
        "party_id": "FIDESZ_KDNP",
        "party_name": "Fidesz–KDNP",
        "party_block": "gov",
        "cand_col": "FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG, KERESZTÉNYDEMOKRATA NÉPPÁRT",
        "list_col": "FIDESZ-KDNP"
    },
    {
        "party_id": "JOBBIK",
        "party_name": "Jobbik",
        "party_block": "opp_radical",
        "cand_col": "JOBBIK MAGYARORSZÁGÉRT MOZGALOM",
        "list_col": "JOBBIK"
    },
    {
        "party_id": "MSZP_PARBESZED",
        "party_name": "MSZP–Párbeszéd",
        "party_block": "opp",
        "cand_col": "MAGYAR SZOCIALISTA PÁRT, PÁRBESZÉD MAGYARORSZÁGÉRT PÁRT",
        "list_col": "MSZP-PÁRBESZÉD"
    },
    {
        "party_id": "DK",
        "party_name": "Demokratikus Koalíció",
        "party_block": "opp",
        "cand_col": "DEMOKRATIKUS KOALÍCIÓ",
        "list_col": "DK"
    },
    {
        "party_id": "LMP",
        "party_name": "Lehet Más a Politika",
        "party_block": "opp",
        "cand_col": "LEHET MÁS A POLITIKA",
        "list_col": "LMP"
    },
    {
        "party_id": "EGYUTT",
        "party_name": "Együtt",
        "party_block": "other",
        "cand_col": "EGYÜTT - A KORSZAKVÁLTÓK PÁRTJA",
        "list_col": "EGYÜTT"
    },
    {
        "party_id": "MOMENTUM",
        "party_name": "Momentum",
        "party_block": "opp",
        "cand_col": "MOMENTUM MOZGALOM",
        "list_col": "MOMENTUM"
    },
    {
        "party_id": "MKKP",
        "party_name": "MKKP",
        "party_block": "other",
        "cand_col": "MAGYAR KÉTFARKÚ KUTYA PÁRT",
        "list_col": "MKKP"
    }
]

# base columns that are repeated for each party
base_cols = [
    "year",
    "county_code",
    "settlement_code",
    "settlement_name",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total"
]

rows = []

# build long-form rows for each main party
for cfg in main_parties:
    cand_col = cfg["cand_col"]
    list_col = cfg["list_col"]
    
    # start from base columns
    tmp = raw_2018[base_cols].copy()
    
    # add party metadata
    tmp["party_id"] = cfg["party_id"]
    tmp["party_name"] = cfg["party_name"]
    tmp["party_block"] = cfg["party_block"]
    
    # candidate and list votes (if column missing, use 0)
    tmp["cand_votes"] = raw_2018.get(cand_col, 0)
    tmp["list_votes"] = raw_2018.get(list_col, 0)
    
    # vote shares, with protection against division by zero
    tmp["cand_share"] = tmp["cand_votes"] / tmp["valid_cand_total"].replace({0: pd.NA})
    tmp["list_share"] = tmp["list_votes"] / tmp["valid_list_total"].replace({0: pd.NA})
    
    rows.append(tmp)

# combine all main parties into one long table
telep_party_results_2018 = pd.concat(rows, ignore_index=True)

# now add an "others" row per settlement using residual votes
# first compute sums of main party votes at settlement level
cand_cols_main = [p["cand_col"] for p in main_parties]
list_cols_main = [p["list_col"] for p in main_parties]

cand_main_sum = raw_2018[cand_cols_main].sum(axis=1)
list_main_sum = raw_2018[list_cols_main].sum(axis=1)

others_cand_votes = (raw_2018["valid_cand_total"] - cand_main_sum).clip(lower=0)
others_list_votes = (raw_2018["valid_list_total"] - list_main_sum).clip(lower=0)

others = raw_2018[base_cols].copy()
others["party_id"] = "OTHERS"
others["party_name"] = "Others"
others["party_block"] = "other"
others["cand_votes"] = others_cand_votes
others["list_votes"] = others_list_votes
others["cand_share"] = others["cand_votes"] / others["valid_cand_total"].replace({0: pd.NA})
others["list_share"] = others["list_votes"] / others["valid_list_total"].replace({0: pd.NA})

# add others to the long table
telep_party_results_2018 = pd.concat(
    [telep_party_results_2018, others],
    ignore_index=True
)


# clean up everything except the final table
del (
    telep_path_2018,
    raw_2018,
    main_parties,
    base_cols,
    rows,
    cfg,
    cand_col,
    list_col,
    tmp,
    cand_cols_main,
    list_cols_main,
    cand_main_sum,
    list_main_sum,
    others_cand_votes,
    others_list_votes,
    others
)


#### 2022

Path to 2022 település-level excel (same file as oevk, but we use the telep sheet).


In [21]:
# path to 2022 település-level excel (same file as oevk, but we use the telep sheet)
telep_path_2022 = (
    data_dir
    / "Választási_eredmények"
    / "2022.04.03._végleges_eredmények"
    / "2022.04.03._valasztasi_eredmenyek_Vox_Populi_adatbazis.xlsx"
)

# read raw excel
raw_2022 = pd.read_excel(telep_path_2022, sheet_name="Települési adatok")

# rename id and total columns to simpler names
raw_2022 = raw_2022.rename(columns={
    "maz": "county_code",
    "taz": "settlement_code",
    "cimt": "settlement_name",
    "Helység.KSH.kódja": "settlement_ksh",
    "Helység.jogállása": "settlement_status",
    "pop6": "pop6",
    # candidate-level totals (egyéni)
    "E.a": "electorate_cand",
    "E.j": "voters_cand",
    "E.n": "valid_cand_total",
    # list-level totals (pártlista)
    "L.a": "electorate_list",
    "L.j": "voters_list",
    "L.n": "valid_list_total"
})

# add year column
raw_2022["year"] = 2022

# main parties at settlement level
# cand_col = candidate votes (E.*)
# list_col = list votes (L.*)
# party_block = gov / opp / opp_radical
main_parties_2022 = [
    {
        "party_id": "FIDESZ_KDNP",
        "party_name": "Fidesz–KDNP",
        "party_block": "gov",
        "cand_col": "E.FK",
        "list_col": "L.FK"
    },
    {
        "party_id": "UNITED_OPP",
        "party_name": "DK–Jobbik–Momentum–MSZP–LMP–Párbeszéd",
        "party_block": "opp",
        "cand_col": "E.EM",
        "list_col": "L.EM"
    },
    {
        "party_id": "MI_HAZANK",
        "party_name": "Mi Hazánk",
        "party_block": "opp_radical",
        "cand_col": "E.MH",
        "list_col": "L.MH"
    }
]

# base columns to carry over for each party
base_cols_2022 = [
    "year",
    "county_code",
    "settlement_code",
    "settlement_name",
    "settlement_ksh",
    "settlement_status",
    "pop6",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total"
]

rows_2022 = []

# build long-form rows for each main party
for cfg in main_parties_2022:
    cand_col = cfg["cand_col"]
    list_col = cfg["list_col"]
    
    # start from base columns
    tmp = raw_2022[base_cols_2022].copy()
    
    # add party metadata
    tmp["party_id"] = cfg["party_id"]
    tmp["party_name"] = cfg["party_name"]
    tmp["party_block"] = cfg["party_block"]
    
    # candidate and list votes (if column missing, use 0)
    tmp["cand_votes"] = raw_2022.get(cand_col, 0)
    tmp["list_votes"] = raw_2022.get(list_col, 0)
    
    # vote shares with protection against division by zero
    tmp["cand_share"] = tmp["cand_votes"] / tmp["valid_cand_total"].replace({0: pd.NA})
    tmp["list_share"] = tmp["list_votes"] / tmp["valid_list_total"].replace({0: pd.NA})
    
    rows_2022.append(tmp)

# combine all main parties into one long table
telep_party_results_2022 = pd.concat(rows_2022, ignore_index=True)

# compute "others" as residual votes at settlement level
cand_cols_main_2022 = [p["cand_col"] for p in main_parties_2022]
list_cols_main_2022 = [p["list_col"] for p in main_parties_2022]

cand_main_sum_2022 = raw_2022[cand_cols_main_2022].sum(axis=1)
list_main_sum_2022 = raw_2022[list_cols_main_2022].sum(axis=1)

others_cand_votes_2022 = (raw_2022["valid_cand_total"] - cand_main_sum_2022).clip(lower=0)
others_list_votes_2022 = (raw_2022["valid_list_total"] - list_main_sum_2022).clip(lower=0)

others_2022 = raw_2022[base_cols_2022].copy()
others_2022["party_id"] = "OTHERS"
others_2022["party_name"] = "Others"
others_2022["party_block"] = "other"
others_2022["cand_votes"] = others_cand_votes_2022
others_2022["list_votes"] = others_list_votes_2022
others_2022["cand_share"] = others_2022["cand_votes"] / others_2022["valid_cand_total"].replace({0: pd.NA})
others_2022["list_share"] = others_2022["list_votes"] / others_2022["valid_list_total"].replace({0: pd.NA})

# add others to the long table
telep_party_results_2022 = pd.concat(
    [telep_party_results_2022, others_2022],
    ignore_index=True
)



# clean up everything except the final table
del (
    telep_path_2022,
    raw_2022,
    main_parties_2022,
    base_cols_2022,
    rows_2022,
    cfg,
    cand_col,
    list_col,
    tmp,
    cand_cols_main_2022,
    list_cols_main_2022,
    cand_main_sum_2022,
    list_main_sum_2022,
    others_cand_votes_2022,
    others_list_votes_2022,
    others_2022
)

#### merge & clean

Build settlement-level lookup from 2022 data.


In [22]:
# build settlement-level lookup from 2022 data
settlement_lookup_2022 = (
    telep_party_results_2022
    .groupby(["county_code", "settlement_code"], as_index=False)
    .agg(
        settlement_ksh=("settlement_ksh", "first"),
        settlement_status=("settlement_status", "first"),
        pop6=("pop6", "first")
    )
)

# merge 2022 settlement attributes into 2018 rows
telep_party_results_2018 = (
    telep_party_results_2018
    .drop(columns=["settlement_ksh", "settlement_status", "pop6"], errors="ignore")
    .merge(
        settlement_lookup_2022,
        on=["county_code", "settlement_code"],
        how="left"
    )
)

# (optional) quick check: are any still missing?
print("missing settlement_ksh in 2018:", telep_party_results_2018["settlement_ksh"].isna().sum())
print("missing settlement_status in 2018:", telep_party_results_2018["settlement_status"].isna().sum())
print("missing pop6 in 2018:", telep_party_results_2018["pop6"].isna().sum())

# now define common column order (based on 2022 structure)
cols_order = [
    "year",
    "county_code",
    "settlement_code",
    "settlement_name",
    "settlement_ksh",
    "settlement_status",
    "pop6",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total",
    "party_id",
    "party_name",
    "party_block",
    "cand_votes",
    "list_votes",
    "cand_share",
    "list_share"
]

# align both dataframes to the same column order
telep_party_results_2018 = telep_party_results_2018[cols_order]
telep_party_results_2022 = telep_party_results_2022[cols_order]

# concat 2018 + 2022
telep_party_results_all = pd.concat(
    [telep_party_results_2018, telep_party_results_2022],
    ignore_index=True
)


# clean up temporary variables
for var in [
    "settlement_lookup_2022",
    "cols_order",
]:
    if var in locals():
        del locals()[var]

missing settlement_ksh in 2018: 0
missing settlement_status in 2018: 0
missing pop6 in 2018: 0


This block prepares the next part of the notebook.


In [23]:
# this block prepares the next part of the notebook.
del(var, telep_party_results_2018, telep_party_results_2022)

This block checks the available column names in the table.


In [24]:
# this block checks the available column names in the table.
telep_party_results_all.columns

Index(['year', 'county_code', 'settlement_code', 'settlement_name',
       'settlement_ksh', 'settlement_status', 'pop6', 'electorate_cand',
       'voters_cand', 'valid_cand_total', 'electorate_list', 'voters_list',
       'valid_list_total', 'party_id', 'party_name', 'party_block',
       'cand_votes', 'list_votes', 'cand_share', 'list_share'],
      dtype='object')

This block shows a quick diagnostic output so we can check the result.


In [25]:
# this block shows a quick diagnostic output so we can check the result.
print(telep_party_results_all["settlement_name"].unique())

['Budapest I. kerület' 'Budapest II. kerület' 'Budapest III. kerület' ...
 'Zalavég' 'Zebecke' 'Tekenye']


This block shows a quick diagnostic output so we can check the result.


In [26]:
# this block shows a quick diagnostic output so we can check the result.
egyedi_telepulesek = telep_party_results_all["settlement_name"].unique()

for telepules in egyedi_telepulesek:
    print(telepules)

print(f"\nÖsszesen: {len(egyedi_telepulesek)} egyedi település.")

Budapest I. kerület
Budapest II. kerület
Budapest III. kerület
Budapest IV. kerület
Budapest V. kerület
Budapest VI. kerület
Budapest VII. kerület
Budapest VIII. kerület
Budapest IX. kerület
Budapest X. kerület
Budapest XI. kerület
Budapest XII. kerület
Budapest XIII. kerület
Budapest XIV. kerület
Budapest XV. kerület
Budapest XVI. kerület
Budapest XVII. kerület
Budapest XVIII. kerület
Budapest XIX. kerület
Budapest XX. kerület
Budapest XXI. kerület
Budapest XXII. kerület
Budapest XXIII. kerület
Abaliget
Adorjás
Almamellék
Almáskeresztúr
Alsómocsolád
Alsószentmárton
Apátvarasd
Aranyosgadány
Ág
Áta
Babarc
Babarcszőlős
Bakonya
Bakóca
Baksa
Baranyahídvég
Baranyajenő
Baranyaszentgyörgy
Basal
Bánfa
Bár
Belvárdgyula
Beremend
Berkesd
Besence
Bezedek
Bicsérd
Bikal
Birján
Bisse
Boda
Bodolyabér
Bogád
Bogádmindszent
Bogdása
Boldogasszonyfa
Borjád
Bosta
Botykapeterd
Bóly
Bükkösd
Bürüs
Csarnóta
Csányoszró
Csebény
Cserdi
Cserkút
Csertő
Csonkamindszent
Cún
Dencsháza
Dinnyeberki
Diósviszló
Drávacsehi


This block prepares the next part of the notebook.


In [27]:
# this block prepares the next part of the notebook.
del(egyedi_telepulesek, telepules)

### European Parliament elections 2019, 2024

Add election_type column to telep_party_results_all.


In [28]:
# add election_type column to telep_party_results_all
telep_party_results_all = telep_party_results_all.copy()
telep_party_results_all["election_type"] = "nat"

#### 2019

Path to 2019 settlement-level excel.


In [29]:
# path to 2019 settlement-level excel
telep_path_2019 = (
    data_dir
    / "Választási_eredmények"
    / "2019.05.26.végleges_eredmények"
    / "telepulesi_eredmenyek_2019.xlsx"
)

# read raw excel
raw_2019 = pd.read_excel(telep_path_2019)

# rename id and total columns to simpler names
raw_2019 = raw_2019.rename(columns={
    "megye_id": "county_code",
    "telepules_id": "settlement_code",
    "megye": "county_name",
    "telepules": "settlement_name",
    "A_választópolgárok_száma": "electorate_list",
    "E_szavazóként_megjelent": "voters_list",
    "N_érvényes": "valid_list_total"
})

# add year column
raw_2019["year"] = 2019

# ep election has only list votes, so candidate-level totals are missing
# keep columns for consistency and set them to na
raw_2019["electorate_cand"] = pd.NA
raw_2019["voters_cand"] = pd.NA
raw_2019["valid_cand_total"] = pd.NA

# these columns do not exist in 2019 ep data, fill with na to match structure
raw_2019["settlement_ksh"] = pd.NA
raw_2019["settlement_status"] = pd.NA
raw_2019["pop6"] = pd.NA

# setup main parties for 2019 ep
# list_col = column with list votes
# party_block = gov / opp / opp_radical
main_parties_2019 = [
    {
        "party_id": "FIDESZ_KDNP",
        "party_name": "Fidesz–KDNP",
        "party_block": "gov",
        "list_col": "fidesz"
    },
    {
        "party_id": "JOBBIK",
        "party_name": "Jobbik",
        "party_block": "opp",
        "list_col": "jobbik"
    },
    {
        "party_id": "MOMENTUM",
        "party_name": "Momentum",
        "party_block": "opp",
        "list_col": "momentum"
    },
    {
        "party_id": "MH",
        "party_name": "Mi Hazánk",
        "party_block": "opp_radical",
        "list_col": "mh"
    },
    {
        "party_id": "MSZP_PARBESZED",
        "party_name": "MSZP–Párbeszéd",
        "party_block": "opp",
        "list_col": "mszp.p"
    },
    {
        "party_id": "LMP",
        "party_name": "Lehet Más a Politika",
        "party_block": "opp",
        "list_col": "lmp"
    },
    {
        "party_id": "DK",
        "party_name": "Demokratikus Koalíció",
        "party_block": "opp",
        "list_col": "dk"
    },
    {
        "party_id": "MP",
        "party_name": "Munkáspárt",
        "party_block": "other",
        "list_col": "mp"
    },
    {
        "party_id": "MKKP",
        "party_name": "MKKP",
        "party_block": "other",
        "list_col": "mkkp"
    }
]

# base columns that are repeated for each party
base_cols_2019 = [
    "year",
    "county_code",
    "settlement_code",
    "settlement_name",
    "settlement_ksh",
    "settlement_status",
    "pop6",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total"
]

rows_2019 = []

# build long-form rows for each main party
for cfg in main_parties_2019:
    list_col = cfg["list_col"]

    tmp = raw_2019[base_cols_2019].copy()

    tmp["party_id"] = cfg["party_id"]
    tmp["party_name"] = cfg["party_name"]
    tmp["party_block"] = cfg["party_block"]

    # no candidate votes at ep -> set to 0
    tmp["cand_votes"] = 0

    # list votes (if column missing, use 0)
    tmp["list_votes"] = raw_2019.get(list_col, 0)

    # candidate share is always na because there is no valid_cand_total
    tmp["cand_share"] = pd.NA

    # list vote share, protect against division by zero
    tmp["list_share"] = tmp["list_votes"] / tmp["valid_list_total"].replace({0: pd.NA})

    rows_2019.append(tmp)

# combine all main parties into one long table
telep_party_results_2019 = pd.concat(rows_2019, ignore_index=True)

# compute residual "others" list votes
list_cols_main_2019 = [p["list_col"] for p in main_parties_2019]

list_main_sum_2019 = raw_2019[list_cols_main_2019].sum(axis=1)

others_list_votes_2019 = (
    raw_2019["valid_list_total"].fillna(0) - list_main_sum_2019.fillna(0)
).clip(lower=0)

others_2019 = raw_2019[base_cols_2019].copy()
others_2019["party_id"] = "OTHERS"
others_2019["party_name"] = "Others"
others_2019["party_block"] = "other"
others_2019["cand_votes"] = 0
others_2019["list_votes"] = others_list_votes_2019
others_2019["cand_share"] = pd.NA
others_2019["list_share"] = (
    others_2019["list_votes"] / others_2019["valid_list_total"].replace({0: pd.NA})
)

# add others to the long table
telep_party_results_2019 = pd.concat(
    [telep_party_results_2019, others_2019],
    ignore_index=True
)

# add election type for clarity
telep_party_results_2019["election_type"] = "ep"

# enforce final column order and add any missing columns as na
final_cols_2019 = [
    "year",
    "county_code",
    "settlement_code",
    "settlement_name",
    "settlement_ksh",
    "settlement_status",
    "pop6",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total",
    "party_id",
    "party_name",
    "party_block",
    "cand_votes",
    "list_votes",
    "cand_share",
    "list_share",
    "election_type"
]

for c in final_cols_2019:
    if c not in telep_party_results_2019.columns:
        telep_party_results_2019[c] = pd.NA

telep_party_results_2019 = telep_party_results_2019[final_cols_2019]

# clean up everything except the final table
for var in [
    "telep_path_2019",
    "raw_2019",
    "main_parties_2019",
    "base_cols_2019",
    "rows_2019",
    "cfg",
    "list_col",
    "tmp",
    "list_cols_main_2019",
    "list_main_sum_2019",
    "others_list_votes_2019",
    "others_2019",
    "final_cols_2019",
]:
    if var in locals():
        del locals()[var]


This block prepares the next part of the notebook.


In [30]:
# this block prepares the next part of the notebook.
del(c, var)

#### 2024

This block builds the 2024 table.


In [31]:
# this block builds the 2024 table.
import pandas as pd
from pathlib import Path

# path to 2024 settlement-level ep excel with percentages
telep_path_2024_share = (
    data_dir
    / "Választási_eredmények"
    / "2024.06.09_végleges_EP_választási_eredmények"
    / "telepulesi_eredmenyek_2024.xlsx"
)

# read raw excel
raw_2024_share = pd.read_excel(telep_path_2024_share)

# normalize column names
raw_2024_share.columns = raw_2024_share.columns.astype(str).str.strip()

# rename id and total columns
raw_2024_share = raw_2024_share.rename(columns={
    "megye_id": "county_code",
    "település_id": "settlement_code",
    "megye_név": "county_name",
    "település_név": "settlement_name",
    "A_választópolgárok_száma": "electorate_list",
    "F_szavazóként_megjelent_a_szavazókörökben": "voters_list",
    "N_érvényes_szavazat": "valid_list_total"
})

# add year
raw_2024_share["year"] = 2024

# add empty columns that exist in nat-election tables but not here
raw_2024_share["settlement_ksh"] = pd.NA
raw_2024_share["settlement_status"] = pd.NA
raw_2024_share["pop6"] = pd.NA

# ep election has only list shares, no candidate totals
raw_2024_share["electorate_cand"] = pd.NA
raw_2024_share["voters_cand"] = pd.NA
raw_2024_share["valid_cand_total"] = pd.NA

# setup main parties for 2024 ep (share version)
main_parties_2024_share = [
    {
        "party_id": "FIDESZ_KDNP",
        "party_name": "Fidesz–KDNP",
        "party_block": "gov",
        "list_col": "Fidesz"
    },
    {
        "party_id": "MI_HAZANK",
        "party_name": "Mi Hazánk",
        "party_block": "opp_radical",
        "list_col": "Mi.Hazánk"
    },
    {
        "party_id": "TISZA",
        "party_name": "TISZA",
        "party_block": "opp",
        "list_col": "TISZA"
    },
    {
        "party_id": "DK_MSZP_P",
        "party_name": "DK–MSZP–P",
        "party_block": "opp",
        "list_col": "DK-MSZP-P"
    },
    {
        "party_id": "LMP",
        "party_name": "LMP",
        "party_block": "other",
        "list_col": "LMP"
    },
    {
        "party_id": "MKKP",
        "party_name": "Kutyapárt",
        "party_block": "other",
        "list_col": "Kutyapárt"
    },
    {
        "party_id": "MEGOLDAS",
        "party_name": "Megoldás",
        "party_block": "other",
        "list_col": "Megoldás"
    },
    {
        "party_id": "KETRK",
        "party_name": "2RK",
        "party_block": "other",
        "list_col": "2RK"
    },
    {
        "party_id": "MOMENTUM",
        "party_name": "Momentum",
        "party_block": "other",
        "list_col": "Momentum"
    },
    {
        "party_id": "JOBBIK",
        "party_name": "Jobbik",
        "party_block": "other",
        "list_col": "Jobbik"
    },
    {
        "party_id": "MINDENKI_M",
        "party_name": "Mindenki Magyarországa",
        "party_block": "other",
        "list_col": "Mindenki.M..N."
    }
]

# base columns repeated for each party
base_cols_2024_share = [
    "year",
    "county_code",
    "settlement_code",
    "settlement_name",
    "settlement_ksh",
    "settlement_status",
    "pop6",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total"
]

rows_2024_share = []

# build long-form rows for each main party
for cfg in main_parties_2024_share:
    list_col = cfg["list_col"]

    tmp = raw_2024_share[base_cols_2024_share].copy()

    tmp["party_id"] = cfg["party_id"]
    tmp["party_name"] = cfg["party_name"]
    tmp["party_block"] = cfg["party_block"]

    # percentage in excel, convert to share in [0, 1]
    share_pct = raw_2024_share.get(list_col, pd.NA)
    tmp["list_share"] = share_pct / 100

    # reconstruct list_votes from share and total valid list votes
    tmp["list_votes"] = tmp["list_share"] * tmp["valid_list_total"]

    tmp["cand_votes"] = pd.NA
    tmp["cand_share"] = pd.NA

    rows_2024_share.append(tmp)

telep_party_results_2024 = pd.concat(rows_2024_share, ignore_index=True)

# residual others share for any omitted lists
list_cols_main_2024 = [p["list_col"] for p in main_parties_2024_share]
share_main_2024 = pd.DataFrame(
    {
        col: pd.to_numeric(raw_2024_share.get(col, 0), errors="coerce").fillna(0.0)
        for col in list_cols_main_2024
    }
) / 100.0
others_list_share_2024 = (1.0 - share_main_2024.sum(axis=1)).clip(lower=0)

others_2024_share = raw_2024_share[base_cols_2024_share].copy()
others_2024_share["party_id"] = "OTHERS"
others_2024_share["party_name"] = "Others"
others_2024_share["party_block"] = "other"
others_2024_share["cand_votes"] = pd.NA
others_2024_share["list_share"] = others_list_share_2024
others_2024_share["list_votes"] = others_2024_share["list_share"] * others_2024_share["valid_list_total"]
others_2024_share["cand_share"] = pd.NA

telep_party_results_2024 = pd.concat(
    [telep_party_results_2024, others_2024_share],
    ignore_index=True
)

telep_party_results_2024["election_type"] = "ep"

# enforce exact column order
col_order = [
    "year",
    "county_code",
    "settlement_code",
    "settlement_name",
    "settlement_ksh",
    "settlement_status",
    "pop6",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total",
    "party_id",
    "party_name",
    "party_block",
    "cand_votes",
    "list_votes",
    "cand_share",
    "list_share",
    "election_type"
]

telep_party_results_2024 = telep_party_results_2024.reindex(columns=col_order)

# clean up temporary variables
for var in [
    "telep_path_2024_share",
    "raw_2024_share",
    "main_parties_2024_share",
    "base_cols_2024_share",
    "rows_2024_share",
    "cfg",
    "list_col",
    "tmp",
    "others_2024_share",
    "col_order",
    "share_pct",
]:
    if var in locals():
        del locals()[var]


This block prepares the next part of the notebook.


In [32]:
# this block prepares the next part of the notebook.
del(var)

This block checks the available column names in the table.


In [33]:
# this block checks the available column names in the table.
telep_party_results_all.columns

Index(['year', 'county_code', 'settlement_code', 'settlement_name',
       'settlement_ksh', 'settlement_status', 'pop6', 'electorate_cand',
       'voters_cand', 'valid_cand_total', 'electorate_list', 'voters_list',
       'valid_list_total', 'party_id', 'party_name', 'party_block',
       'cand_votes', 'list_votes', 'cand_share', 'list_share',
       'election_type'],
      dtype='object')

#### merge & clean

Lookup table from nat elections (2018 + 2022).


In [34]:
# lookup table from nat elections (2018 + 2022)
settlement_lookup = (
    telep_party_results_all
    .groupby(["county_code", "settlement_code"], as_index=False)
    .agg(
        settlement_ksh=("settlement_ksh", "first"),
        settlement_status=("settlement_status", "first"),
        pop6=("pop6", "first")
    )
)

# desired column order
col_order = [
    "year",
    "county_code",
    "settlement_code",
    "settlement_name",
    "settlement_ksh",
    "settlement_status",
    "pop6",
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total",
    "party_id",
    "party_name",
    "party_block",
    "cand_votes",
    "list_votes",
    "cand_share",
    "list_share",
    "election_type"
]

# enrich 2019 and 2024 tables
for name in ["telep_party_results_2019", "telep_party_results_2024"]:
    if name in globals():
        df = globals()[name]

        # drop old versions of these columns if they exist
        df = df.drop(columns=["settlement_ksh", "settlement_status", "pop6"], errors="ignore")

        # merge lookup info
        df = df.merge(
            settlement_lookup,
            on=["county_code", "settlement_code"],
            how="left"
        )

        # enforce column order
        df = df.reindex(columns=col_order)

        globals()[name] = df

# optional check
print("2019 missing settlement_ksh:", telep_party_results_2019["settlement_ksh"].isna().sum())
print("2024 missing settlement_ksh:", telep_party_results_2024["settlement_ksh"].isna().sum())

# clean up helpers
for var in ["settlement_lookup", "col_order", "name", "df"]:
    if var in locals():
        del locals()[var]

del(var)        

2019 missing settlement_ksh: 0
2024 missing settlement_ksh: 0


Check the column order: verify that the three tables have exactly the same columns in the same order.


In [35]:
# check the column order
# check if the three tables have exactly the same columns in the same order

# reference = nat elections
ref_cols = list(telep_party_results_all.columns)

# 2019 vs all
cols_2019 = list(telep_party_results_2019.columns)
print("2019 same cols and order as all?:", cols_2019 == ref_cols)

# 2024 vs all
cols_2024 = list(telep_party_results_2024.columns)
print("2024 same cols and order as all?:", cols_2024 == ref_cols)

# optional: show where they first differ
def first_diff(a, b):
    n = min(len(a), len(b))
    for i in range(n):
        if a[i] != b[i]:
            return i, a[i], b[i]
    if len(a) != len(b):
        return n, None, None
    return None

print("first diff 2019 vs all:", first_diff(ref_cols, cols_2019))
print("first diff 2024 vs all:", first_diff(ref_cols, cols_2024))

# clean up helper variables
for var in ["ref_cols", "cols_2019", "cols_2024", "first_diff", "var"]:
    if var in locals():
        del locals()[var]

2019 same cols and order as all?: True
2024 same cols and order as all?: True
first diff 2019 vs all: None
first diff 2024 vs all: None


Merging: ensure 2019 and 2024 have exactly the same columns (and order) as the full table.


In [36]:
# merging
# ensure 2019 and 2024 have exactly the same columns (and order) as the all table
common_cols = list(telep_party_results_all.columns)

telep_party_results_2019 = telep_party_results_2019.reindex(columns=common_cols)
telep_party_results_2024 = telep_party_results_2024.reindex(columns=common_cols)

# append 2019 and 2024 rows to the all table
telep_party_results_all = pd.concat(
    [
        telep_party_results_all,
        telep_party_results_2019,
        telep_party_results_2024,
    ],
    ignore_index=True
)

# clean up helper variables
for var in ["common_cols"]:
    if var in locals():
        del locals()[var]
del(var)

/var/folders/m_/_l_ts7n95x971syc73dnxsk00000gn/T/ipykernel_9260/4229580807.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  telep_party_results_all = pd.concat(


Check number of settlements per year in the full table.


In [37]:
# check
# number of settlements per year in the full table
print(
    telep_party_results_all
    .groupby("year")["settlement_name"]
    .nunique()
)

# number of rows per year in the full table
print(
    telep_party_results_all["year"].value_counts()
)

# check how many party_id values per settlement per year
print(
    telep_party_results_all
    .groupby(["year", "settlement_code"])["party_id"]
    .nunique()
    .value_counts()
)

# number of distinct parties per year in the full table
print(
    telep_party_results_all
    .groupby("year")["party_id"]
    .nunique()
)

year
2018    3177
2019    3177
2022    3177
2024    3177
Name: settlement_name, dtype: int64
year
2024    38124
2019    31770
2018    28593
2022    12708
Name: count, dtype: int64
party_id
9     358
10    358
4     358
12    358
Name: count, dtype: int64
year
2018     9
2019    10
2022     4
2024    12
Name: party_id, dtype: int64


For 358 (year, settlement_code) combinations, there are 9 different party_id values. For 358 combinations, there are 10 parties. For 358 combinations, there are 4 parties.


In [38]:
# For 358 (year, settlement_code) combinations, there are 9 different party_id values.
# For 358 combinations, there are 10 parties.
# For 358 combinations, there are 4 parties.
# For 358 combinations, there are 12 parties.

Filtering unnecessary rows.


In [39]:
# filtering unnecessary rows
telep_party_results_all["settlement_name"].unique() # district numbering is not good!!!

array(['Budapest I. kerület', 'Budapest II. kerület',
       'Budapest III. kerület', ..., 'Budapest 21. kerület',
       'Budapest 22. kerület', 'Budapest 23. kerület'],
      shape=(3223,), dtype=object)

Renaming helper to normalize all Budapest district name variants.


In [40]:
# renaming helper
# helper to normalize all Budapest district name variants
def normalize_budapest_name(name):
    if not isinstance(name, str):
        return name

    s = name.strip()

    # only touch strings starting with "Budapest"
    if not s.startswith("Budapest"):
        return s

    # roman numeral map for districts
    roman_map = {
        "I": 1,
        "II": 2,
        "III": 3,
        "IV": 4,
        "V": 5,
        "VI": 6,
        "VII": 7,
        "VIII": 8,
        "IX": 9,
        "X": 10,
        "XI": 11,
        "XII": 12,
        "XIII": 13,
        "XIV": 14,
        "XV": 15,
        "XVI": 16,
        "XVII": 17,
        "XVIII": 18,
        "XIX": 19,
        "XX": 20,
        "XXI": 21,
        "XXII": 22,
        "XXIII": 23,
    }

    # case 1: arabic with optional leading zeros, with or without dot before "kerület"
    # e.g. "Budapest 01. kerület", "Budapest 03 kerület", "Budapest 10. kerület"
    m = re.match(r"^Budapest\s+0*(\d{1,2})\.?\s+kerület$", s)
    if m:
        num = int(m.group(1))
        return f"Budapest {num}. kerület"

    # case 2: roman numerals with "kerület"
    # e.g. "Budapest IX. kerület", "Budapest XXIII kerület"
    m = re.match(r"^Budapest\s+([IVXLCDM]+)\.?\s+kerület$", s)
    if m:
        roman = m.group(1)
        num = roman_map.get(roman)
        if num is not None:
            return f"Budapest {num}. kerület"

    # case 3: roman numerals without "kerület"
    # e.g. "Budapest XXIII"
    m = re.match(r"^Budapest\s+([IVXLCDM]+)$", s)
    if m:
        roman = m.group(1)
        num = roman_map.get(roman)
        if num is not None:
            return f"Budapest {num}. kerület"

    # everything else stays as is
    return s

# apply to settlement_name
telep_party_results_all["settlement_name"] = (
    telep_party_results_all["settlement_name"]
    .astype(str)
    .apply(normalize_budapest_name)
)

This block prepares the next part of the notebook.


In [41]:
# this block prepares the next part of the notebook.

telep_party_results_all["settlement_name"].unique() # now it's good: same 3177 unique settlement names globally, not just within the yearly grouping

array(['Budapest 1. kerület', 'Budapest 2. kerület',
       'Budapest 3. kerület', ..., 'Zalavég', 'Zebecke', 'Tekenye'],
      shape=(3177,), dtype=object)

This block prepares the next part of the notebook.


In [42]:
# this block prepares the next part of the notebook.
del(telep_party_results_2019, telep_party_results_2024)

### Settlement level statistics

Helper function to load and clean one ksh settlement-level excel.


In [43]:
# helper function to load and clean one ksh settlement-level excel
def load_ksh_telep_file(xlsx_path: Path) -> pd.DataFrame:
    # read excel
    df = pd.read_excel(xlsx_path)

    # rename columns to simpler english-like names
    df = df.rename(columns={
        "Terület": "settlement_name",
        "Lakónépesség száma az év végén (a népszámlálás végleges adataiból továbbvezetett adat) (fő)": "pop_total",
        "Lakónépességből a 0-14 évesek száma az év végén (fő)": "pop_0_14",
        "Lakónépességből a 65 éves és idősebbek száma az év végén (fő)": "pop_65_plus",
        "Állandó népesség száma (fő)": "pop_perm",
        "Állandó népességből a 0-14 éves nők száma (fő)": "pop_perm_0_14_female",
        "Állandó népességből a 65-X éves nők száma (fő)": "pop_perm_65_plus_female",
        "Állandó népességből a 0-14 éves férfiak száma (fő)": "pop_perm_0_14_male",
        "Állandó népességből a 65-X éves férfiak száma (fő)": "pop_perm_65_plus_male",
        "Állandó népességből a 0-24 évesek száma (fő)": "pop_perm_0_24",
        "A helyi önkormányzatok összes költségvetési bevétele (1000 Ft)": "budget_revenue",
        "A helyi önkormányzatok összes költségvetési kiadása (1000 Ft)": "budget_expenditure",
        "Épített lakások száma (db)": "dwellings_built",
        "Az év folyamán megszűnt lakások száma (db)": "dwellings_terminated",
        "Az év folyamán a helyi önkormányzat által épített lakások száma (db)": "dwellings_built_muni",
        "Az év folyamán saját használatra épített lakások száma (db)": "dwellings_built_private",
        "Háziorvosok száma (fő)": "gps_doctors",
        "Házi gyermekorvosok száma (fő)": "pediatric_doctors",
        "Személygépkocsik száma az üzemeltető lakhelye szerint (db)": "cars",
        "Internet-előfizetések egyéb kapcsolaton keresztül (LAN, bérelt vonal, modemes, vezeték nélküli (mobilinternet nélkül), stb.) 2016-tól (db)": "internet_other",
        "Helyhez kötött televíziószolgáltatás előfizetéseinek száma (db)": "tv_subs",
        "Magyarországon első alkalommal forgalomba helyezett gépjárművek száma (db)": "cars_first_reg",
        "Postai szolgáltatóhelyek száma (db)": "post_offices",
        "Internet-előfizetések száma (db)": "internet_total",
        "Internet-előfizetések xDSL hálózaton (db)": "internet_xdsl",
        "Internet-előfizetések kábeltelevízió-hálózaton (db)": "internet_cable",
        "Internet-előfizetések optikai hálózaton (db)": "internet_optical",
        "Nyilvántartott álláskeresők száma összesen (fő)": "unemployed_total",
        "180 napon túli nyilvántartott álláskeresők száma összesen (fő)": "unemployed_180_plus",
        "Egyetemi végzettségű nyilvántartott álláskeresők száma (fő)": "unemployed_univ",
        "Fizikai foglalkozású nyilvántartott álláskeresők száma (fő)": "unemployed_bluecollar",
        "Nyilvántartott pályakezdő álláskeresők száma (fő)": "unemployed_young",
        "Egy éven túl nyilvántartott álláskeresők száma összesen (fő)": "unemployed_365_plus",
        "Borvidékek, 2019": "wine_region",
        "Statisztikai nagyrégió": "stat_region",
        "Agglomeráción belüli települések besorolása, 2003": "agglomeration_category_2003",
        "Agglomeráció típusa, 2003": "agglomeration_type_2003",
        "Járások": "district",
        "Agglomerációk, 2003": "agglomeration_2003",
        "Megyék - általános területi beosztás": "county_general",
        "Települések jogállása - összevont verzió, 1994": "settlement_status_1994",
        "Kedvezményezett települések  kódjának nómenklatúrája": "fav_settlement_code",
        "Borrégiók, 2019": "wine_region_2",
        "Kedvezményezett járások kódjának nómenklatúrája": "fav_district_code",
        "Települések": "settlement_type",
        "Turisztikai régiók": "tourism_region",
        "Világörökségi helyszín védőövezete": "world_heritage_buffer",
        "Tervezési statisztikai régiók": "planning_region",
        "Az EU NUTS rendszere 2016": "nuts_2016",
        "Szabad vállalkozási zónák": "free_enterprise_zone",
        "Területfejlesztési szempontból kiemelt térségek": "dev_priority_area",
        "Települések megyeszékhely jellege, 1991": "county_seat_status_1991",
        "Járási székhely": "district_seat",
        "Települések jogállása - bővített verzió, 2005": "settlement_status_2005",
        "Települések lakónépesség-nagyságkategóriái": "settlement_pop_size_cat",
        "Nemzeti parkok": "national_park",
        "Év": "year",
    })

    # clean settlement names
    df["settlement_name"] = df["settlement_name"].astype(str).str.strip()

    # drop rows with aggregated or non-allocatable units
    drop_mask = (
        df["settlement_name"].str.contains("nem bontható adatai", case=False, na=False)
        | df["settlement_name"].str.contains("nem bontható adatok", case=False, na=False)
        | df["settlement_name"].str.contains("kívüli tevékenység", case=False, na=False)
        | (df["settlement_name"].str.strip() == "Budapest")
    )

    df = df.loc[~drop_mask].reset_index(drop=True)

    return df

# list of files to combine (your actual paths)
telep_files = [
    data_dir.parent / "KSH" / "telepules_szintu" / "2016 (18as választáshoz)" / "2016_merged.xlsx",
    data_dir.parent / "KSH" /"telepules_szintu" / "2017 (19es ep választáshoz)" / "2017_merged.xlsx",
    data_dir.parent / "KSH" /"telepules_szintu" / "2020 (22es választáshoz)" / "2020_merged.xlsx",
    data_dir.parent / "KSH" /"telepules_szintu" / "2022 (24es ep választáshoz)" / "2022_merged.xlsx",
    data_dir.parent / "KSH" /"telepules_szintu" / "2024 (26os választáshoz)" / "2024_merged.xlsx",
]

# load and clean all tables
dfs_telep = []
for fp in telep_files:
    df_tmp = load_ksh_telep_file(fp)
    dfs_telep.append(df_tmp)

# stack them into one big dataframe
telep_merged_all = pd.concat(dfs_telep, ignore_index=True)

# clean up temporary variables
for var in ["load_ksh_telep_file", "telep_files", "dfs_telep", "fp", "df_tmp"]:
    if var in locals():
        del locals()[var]
del(var)

Define which election the data can be used (row by row) make a copy to be safe.


In [44]:
# define which election the data can be used (row by row)
# make a copy to be safe
telep_merged_all = telep_merged_all.copy()

# new column: which election year this t-2 data can be used for
telep_merged_all["target_election_year"] = telep_merged_all["year"] + 2

# if you want to be sure it's integer type
telep_merged_all["target_election_year"] = telep_merged_all["target_election_year"].astype("Int64")

# check
telep_merged_all["target_election_year"].unique()

<IntegerArray>
[2018, 2019, 2022, 2024, 2026]
Length: 5, dtype: Int64

### County level statistics

Path to territorial labour income excel.


In [45]:
# path to territorial labour income excel
teruleti_path = (
    data_dir.parent
    / "KSH"
    / "teruleti_munkval_jov"
    / "Területi_munkavállalói_jövedelem.xlsx"
)

# read modified sheet
teruleti_income = pd.read_excel(
    teruleti_path,
    sheet_name="modified"
)

# ensure period column is numeric and create target election year as period + 3
teruleti_income["Időszak"] = pd.to_numeric(
    teruleti_income["Időszak"],
    errors="coerce"
)
teruleti_income["target_election_year"] = teruleti_income["Időszak"] + 3


# rename main columns
teruleti_income = teruleti_income.rename(columns={
    "Terület": "county_general",
    "Területi munkavállalói jövedelem (millió Ft)": "labour_income_million_huf_t_3"
})

# clean up temporary variables
for var in ["teruleti_path"]:
    if var in locals():
        del locals()[var]
del(var)

#### cpi handling

This block handles the CPI series.


In [46]:
# this block handles the CPI series.
import pandas as pd
from pathlib import Path

# read cpi from world bank file (2010 = 100)
cpi_path = (
    data_dir.parent
    / "KSH"
    / "cpi_worldbank"
    / "CPI_worldbank.xlsx"
)

# read modified sheet
cpi_raw = pd.read_excel(cpi_path, sheet_name="modified")

# standardize column names
cpi_raw.columns = cpi_raw.columns.astype(str).str.strip()

# keep only year and cpi columns, drop missing and duplicates
cpi = (
    cpi_raw[["year", "cpi"]]
    .dropna(subset=["year", "cpi"])
    .drop_duplicates(subset=["year"])
    .copy()
)

# make sure year columns are of the same type for merging
cpi["year"] = cpi["year"].astype(int)
teruleti_income["target_election_year"] = teruleti_income["target_election_year"].astype(int)

# merge cpi to teruleti_income by target_election_year
teruleti_income = teruleti_income.merge(
    cpi,
    left_on="Időszak",
    right_on="year",
    how="left"
)

# drop the duplicate year column from cpi if not needed
teruleti_income = teruleti_income.drop(columns=["year"])

# create real income column in 2010 prices
teruleti_income["labor_income_million_huf_t_3_real2010"] = (
    teruleti_income["labour_income_million_huf_t_3"]
    * 100
    / teruleti_income["cpi"]
)

# drop original period column
teruleti_income = teruleti_income.drop(columns=["Időszak"])

del(cpi, cpi_path, cpi_raw)

#### merging the county level to the settlement level

Normalize county_general by keeping only the part after the dash.


In [47]:
# normalize county_general by keeping only the part after the dash
telep_merged_all["county_general"] = (
    telep_merged_all["county_general"]
    .astype(str)
    .str.split("-", n=1)
    .str[-1]
    .str.strip()
)

# merge territorial income onto the large telep table
# many telep_merged_all rows will match the same (county_general, target_election_year),
# which is fine because we use a left join
telep_merged_all = telep_merged_all.merge(
    teruleti_income,
    on=["county_general", "target_election_year"],
    how="left"
)
del(teruleti_income)

### Media consumption data

#### for 2022 election

Read excel with two header rows from the "Report data" sheet.


In [48]:
# read excel with two header rows from the "Report data" sheet
gemius_path = (
    data_dir.parent
    / "Gemius"
    / "2022_Audience_Hungary_2026-03-04.xlsx"
)

df_raw = pd.read_excel(
    gemius_path,
    sheet_name="Report data",
    header=[0, 1]
)

# standardize both header levels to clean strings
df_raw.columns = pd.MultiIndex.from_tuples(
    [(str(a).strip(), str(b).strip()) for a, b in df_raw.columns]
)

# keep id columns separately for clarity
id_media = ("Media channel", "Media channel")
id_county = ("County", "County")
id_place = ("Place of living size (inhabitants)", "Place of living size (inhabitants)")

id_cols = [id_media, id_county, id_place]

# copy to a working dataframe
df = df_raw.copy()

# list of metrics we care about
metrics = ["Real users", "Views", "Time per view [s]"]

# find all metric columns and convert them from strings with spaces/commas to numeric
metric_cols = [c for c in df.columns if c[1] in metrics]

for col in metric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

df[metric_cols] = df[metric_cols].apply(pd.to_numeric, errors="coerce")

# define time windows by months
windows = {
    "w1": ["2021-04", "2021-05", "2021-06", "2021-07", "2021-08", "2021-09"],
    "w2": ["2021-10", "2021-11", "2021-12", "2022-01"],
    "w3": ["2022-02", "2022-03"],
}

# map metric names to simple base names
metric_base_names = {
    "Real users": "real_users",
    "Views": "views",
    "Time per view [s]": "time_per_view_s",
}

# start output with id columns
gemius_22 = pd.DataFrame({
    "media_channel": df[id_media],
    "county": df[id_county],
    "place_size": df[id_place],
})

# for each window and each metric compute row wise mean across months
for win_label, months in windows.items():
    for metric in metrics:
        cols = [
            col for col in df.columns
            if (col[0] in months and col[1] == metric)
        ]

        if not cols:
            continue

        new_col_name = f"{metric_base_names[metric]}_{win_label}"

        gemius_22[new_col_name] = df.loc[:, cols].mean(axis=1)

# clean up temporary variables, keep only gemius_22
for var in [
    "gemius_path",
    "df_raw",
    "df",
    "id_media",
    "id_county",
    "id_place",
    "id_cols",
    "metrics",
    "metric_cols",
    "windows",
    "metric_base_names",
    "win_label",
    "months",
    "metric",
    "cols",
]:
    if var in locals():
        del locals()[var]
del var, new_col_name

# keep only rows where media_channel is not "All"
gemius_22 = gemius_22[gemius_22["media_channel"] != "All"]

Work on a copy of the 2022 audience table.


In [49]:
# work on a copy of the 2022 audience table
gemius_tmp = gemius_22.copy()

# create a clean media name that is safe to use in column names
gemius_tmp["media_clean"] = (
    gemius_tmp["media_channel"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^0-9a-z]+", "_", regex=True)
    .str.strip("_")
)

# id columns that stay as row identifiers
id_cols = ["county", "place_size"]

# save full index of all county–place_size combinations so that rows with all-nan are also kept
full_index = (
    gemius_tmp[id_cols]
    .drop_duplicates()
    .set_index(id_cols)
    .index
)

# metric columns are everything except ids and media labels
metric_cols = [
    c
    for c in gemius_tmp.columns
    if c not in ["media_channel", "media_clean"] + id_cols
]

# full list of media values to enforce as columns, even if all values are na
all_media = (
    gemius_tmp["media_clean"]
    .dropna()
    .unique()
)

frames = []

# pivot each metric separately, then concatenate
for col in metric_cols:
    # rows are county + place size, columns are media_clean
    pivot = gemius_tmp.pivot_table(
        index=id_cols,
        columns="media_clean",
        values=col,
        aggfunc="mean"
    )

    # make sure we keep a column for every media, even if all values are na
    pivot = pivot.reindex(columns=all_media)

    # ensure we have all county–place_size combinations, even if all values are nan
    pivot = pivot.reindex(full_index)

    # flatten column names to pattern "<metric>_<media>"
    pivot.columns = [f"{col}_{m}" for m in pivot.columns]

    frames.append(pivot)

# combine all metrics back together
gemius_22_wide = pd.concat(frames, axis=1).reset_index()

# clean up temporary variables
for var in [
    "gemius_tmp",
    "id_cols",
    "metric_cols",
    "frames",
    "pivot",
    "col",
    "all_media",
    "full_index",
]:
    if var in locals():
        del locals()[var]
del var, gemius_22

# filtering the unnecessary "All" rows
gemius_22_wide = gemius_22_wide[gemius_22_wide["county"] != "All"]
gemius_22_wide = gemius_22_wide[gemius_22_wide["place_size"] != "All"]

#### for 2026 election

Read excel with two header rows from the "Report data" sheet.


In [50]:
# read excel with two header rows from the "Report data" sheet
gemius_path = (
    data_dir.parent
    / "Gemius"
    / "2026_Audience_Hungary_2026-03-05.xlsx"
)

df_raw = pd.read_excel(
    gemius_path,
    sheet_name="Report data",
    header=[0, 1]
)

# standardize both header levels to clean strings
df_raw.columns = pd.MultiIndex.from_tuples(
    [(str(a).strip(), str(b).strip()) for a, b in df_raw.columns]
)

# keep id columns separately for clarity
id_media = ("Media channel", "Media channel")
id_county = ("County", "County")
id_place = ("Place of living size (inhabitants)", "Place of living size (inhabitants)")

id_cols = [id_media, id_county, id_place]

# copy to a working dataframe
df = df_raw.copy()

# list of metrics we care about
metrics = ["Real users", "Views", "Time per view [s]"]

# find all metric columns and convert them from strings with spaces/commas to numeric
metric_cols = [c for c in df.columns if c[1] in metrics]

for col in metric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

df[metric_cols] = df[metric_cols].apply(pd.to_numeric, errors="coerce")

# define time windows by months for the 2026 election
windows = {
    "w1": ["2025-04", "2025-05", "2025-06", "2025-07", "2025-08", "2025-09"],
    "w2": ["2025-10", "2025-11", "2025-12", "2026-01"],
    "w3": ["2026-02", "2026-03"],
}

# map metric names to simple base names
metric_base_names = {
    "Real users": "real_users",
    "Views": "views",
    "Time per view [s]": "time_per_view_s",
}

# start output with id columns
gemius_26 = pd.DataFrame({
    "media_channel": df[id_media],
    "county": df[id_county],
    "place_size": df[id_place],
})

# for each window and each metric compute row wise mean across months
for win_label, months in windows.items():
    for metric in metrics:
        cols = [
            col for col in df.columns
            if (col[0] in months and col[1] == metric)
        ]

        if not cols:
            continue

        new_col_name = f"{metric_base_names[metric]}_{win_label}"

        gemius_26[new_col_name] = df.loc[:, cols].mean(axis=1)

# clean up temporary variables, keep only gemius_26
for var in [
    "gemius_path",
    "df_raw",
    "df",
    "id_media",
    "id_county",
    "id_place",
    "id_cols",
    "metrics",
    "metric_cols",
    "windows",
    "metric_base_names",
    "win_label",
    "months",
    "metric",
    "cols",
]:
    if var in locals():
        del locals()[var]
del var, new_col_name

# keep only rows where media_channel is not "All"
gemius_26 = gemius_26[gemius_26["media_channel"] != "All"]

Work on a copy of the 2026 audience table.


In [51]:
# work on a copy of the 2026 audience table
gemius_tmp = gemius_26.copy()

# create a clean media name that is safe to use in column names
gemius_tmp["media_clean"] = (
    gemius_tmp["media_channel"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^0-9a-z]+", "_", regex=True)
    .str.strip("_")
)

# id columns that stay as row identifiers
id_cols = ["county", "place_size"]

# metric columns are everything except ids and media labels
metric_cols = [
    c
    for c in gemius_tmp.columns
    if c not in ["media_channel", "media_clean"] + id_cols
]

# full list of media values to enforce as columns even if all values are na
all_media = (
    gemius_tmp["media_clean"]
    .dropna()
    .unique()
)

frames = []

# pivot each metric separately then concatenate
for col in metric_cols:
    # rows are county and place size, columns are media_clean
    pivot = gemius_tmp.pivot_table(
        index=id_cols,
        columns="media_clean",
        values=col,
        aggfunc="mean",
        dropna=False  # keep rows even if all values are na
    )

    # keep a column for every media even if all values are na
    pivot = pivot.reindex(columns=all_media)

    # flatten column names to pattern "<metric>_<media>"
    pivot.columns = [f"{col}_{m}" for m in pivot.columns]

    frames.append(pivot)

# combine all metrics back together
gemius_26_wide = pd.concat(frames, axis=1).reset_index()

# clean up temporary variables
for var in [
    "gemius_tmp",
    "id_cols",
    "metric_cols",
    "frames",
    "pivot",
    "col",
    "all_media",
]:
    if var in locals():
        del locals()[var]
del var, gemius_26

# filter out unnecessary all rows
gemius_26_wide = gemius_26_wide[gemius_26_wide["county"] != "All"]
gemius_26_wide = gemius_26_wide[gemius_26_wide["place_size"] != "All"]

#### for 2024 EP election

Read excel with two header rows from the "Report data" sheet.


In [52]:
# read excel with two header rows from the "Report data" sheet
gemius_path = (
    data_dir.parent
    / "Gemius"
    / "2024_Audience_Hungary_2026-03-04.xlsx"
)

df_raw = pd.read_excel(
    gemius_path,
    sheet_name="Report data",
    header=[0, 1]
)

# standardize both header levels to clean strings
df_raw.columns = pd.MultiIndex.from_tuples(
    [(str(a).strip(), str(b).strip()) for a, b in df_raw.columns]
)

# keep id columns separately for clarity
id_media = ("Media channel", "Media channel")
id_county = ("County", "County")
id_place = ("Place of living size (inhabitants)", "Place of living size (inhabitants)")

id_cols = [id_media, id_county, id_place]

# copy to a working dataframe
df = df_raw.copy()

# list of metrics we care about
metrics = ["Real users", "Views", "Time per view [s]"]

# find all metric columns and convert them from strings with spaces/commas to numeric
metric_cols = [c for c in df.columns if c[1] in metrics]

for col in metric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

df[metric_cols] = df[metric_cols].apply(pd.to_numeric, errors="coerce")

# define time windows by months for the 2024 election
# w1: 2023-06 ... 2023-11
# w2: 2023-12 ... 2024-03
# w3: 2024-04 ... 2024-05
windows = {
    "w1": ["2023-06", "2023-07", "2023-08", "2023-09", "2023-10", "2023-11"],
    "w2": ["2023-12", "2024-01", "2024-02", "2024-03"],
    "w3": ["2024-04", "2024-05"],
}

# map metric names to simple base names
metric_base_names = {
    "Real users": "real_users",
    "Views": "views",
    "Time per view [s]": "time_per_view_s",
}

# start output with id columns
gemius_24 = pd.DataFrame({
    "media_channel": df[id_media],
    "county": df[id_county],
    "place_size": df[id_place],
})

# for each window and each metric compute row wise mean across months
for win_label, months in windows.items():
    for metric in metrics:
        cols = [
            col for col in df.columns
            if (col[0] in months and col[1] == metric)
        ]

        if not cols:
            continue

        new_col_name = f"{metric_base_names[metric]}_{win_label}"

        gemius_24[new_col_name] = df.loc[:, cols].mean(axis=1)

# clean up temporary variables, keep only gemius_24
for var in [
    "gemius_path",
    "df_raw",
    "df",
    "id_media",
    "id_county",
    "id_place",
    "id_cols",
    "metrics",
    "metric_cols",
    "windows",
    "metric_base_names",
    "win_label",
    "months",
    "metric",
    "cols",
]:
    if var in locals():
        del locals()[var]
del var, new_col_name

# keep only rows where media_channel is not "All"
gemius_24 = gemius_24[gemius_24["media_channel"] != "All"]

Work on a copy of the 2024 audience table.


In [53]:
# work on a copy of the 2024 audience table
gemius_tmp = gemius_24.copy()

# create a clean media name that is safe to use in column names
gemius_tmp["media_clean"] = (
    gemius_tmp["media_channel"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^0-9a-z]+", "_", regex=True)
    .str.strip("_")
)

# id columns that stay as row identifiers
id_cols = ["county", "place_size"]

# metric columns are everything except ids and media labels
metric_cols = [
    c
    for c in gemius_tmp.columns
    if c not in ["media_channel", "media_clean"] + id_cols
]

# full list of media values to enforce as columns even if all values are na
all_media = (
    gemius_tmp["media_clean"]
    .dropna()
    .unique()
)

frames = []

# pivot each metric separately then concatenate
for col in metric_cols:
    # rows are county and place size, columns are media_clean
    pivot = gemius_tmp.pivot_table(
        index=id_cols,
        columns="media_clean",
        values=col,
        aggfunc="mean",
        dropna=False  # keep rows even if all values are na
    )

    # keep a column for every media even if all values are na
    pivot = pivot.reindex(columns=all_media)

    # flatten column names to pattern "<metric>_<media>"
    pivot.columns = [f"{col}_{m}" for m in pivot.columns]

    frames.append(pivot)

# combine all metrics back together
gemius_24_wide = pd.concat(frames, axis=1).reset_index()

# clean up temporary variables
for var in [
    "gemius_tmp",
    "id_cols",
    "metric_cols",
    "frames",
    "pivot",
    "col",
    "all_media",
]:
    if var in locals():
        del locals()[var]
del var, gemius_24

# filter out unnecessary all rows
gemius_24_wide = gemius_24_wide[gemius_24_wide["county"] != "All"]
gemius_24_wide = gemius_24_wide[gemius_24_wide["place_size"] != "All"]

#### merge them

Before this, it's important to handle the county names in `telep_merged_all` and ensure `county_general` is a string.


In [54]:
# before this, it's important to handle the county names in telep_merged_all
# ensure county_general is string
telep_merged_all["county_general"] = telep_merged_all["county_general"].astype(str)

# 1) where settlement_name starts with "Budapest", force county_general = "Budapest"
bp_mask = telep_merged_all["settlement_name"].astype(str).str.startswith("Budapest")
telep_merged_all.loc[bp_mask, "county_general"] = "Budapest"

# 2) remove " vármegye" suffix from all county_general values
telep_merged_all["county_general"] = (
    telep_merged_all["county_general"]
    .str.replace(r"\s*vármegye$", "", regex=True)
    .str.strip()
)
del bp_mask

Add year column to each wide table.


In [55]:
# add year column to each wide table
gemius_22_wide["year"] = 2022
gemius_24_wide["year"] = 2024
gemius_26_wide["year"] = 2026

# stack the three wide tables on top of each other
gemius_wide_all = pd.concat(
    [gemius_22_wide, gemius_24_wide, gemius_26_wide],
    ignore_index=True
)

# clean up the original wide tables
for var in ["gemius_22_wide", "gemius_24_wide", "gemius_26_wide"]:
    if var in locals():
        del locals()[var]
del var

Make sure population column is numeric.


In [56]:
# make sure population column is numeric
telep_merged_all["pop_total"] = pd.to_numeric(
    telep_merged_all["pop_total"],
    errors="coerce"
)

# define function to map population to gemius place_size buckets
def map_place_size(row):
    county = str(row.get("county_general", ""))
    pop = row.get("pop_total")

    # special case for budapest
    if county == "Budapest":
        return "Budapest"

    # if population is missing, cannot classify
    if pd.isna(pop):
        return pd.NA

    # apply gemius buckets
    if pop < 5000:
        return "Town: 0-5k"
    elif pop < 10000:
        return "Town: 5k-10k"
    elif pop < 20000:
        return "Town: 10k- 20k"
    elif pop < 50000:
        return "Town: 20k-50k"
    elif pop < 100000:
        return "Town: 50k-100k"
    else:
        return "City: 100k+"

# create place_size category to match gemius_wide_all
telep_merged_all["place_size"] = telep_merged_all.apply(
    map_place_size,
    axis=1
)

# make sure year types align
gemius_wide_all["year"] = pd.to_numeric(gemius_wide_all["year"], errors="coerce")
# fix a known Gemius county-name encoding mismatch before the join
gemius_wide_all["county"] = (
    gemius_wide_all["county"]
    .astype("string")
    .str.replace("Gyõr-Moson-Sopron", "Győr-Moson-Sopron", regex=False)
)
telep_merged_all["target_election_year"] = pd.to_numeric(
    telep_merged_all["target_election_year"],
    errors="coerce"
)

# merge gemius_wide_all onto telep_merged_all
# match by county + place_size + election year
telep_merged_all = telep_merged_all.merge(
    gemius_wide_all,
    left_on=["county_general", "place_size", "target_election_year"],
    right_on=["county", "place_size", "year"],
    how="left",
    suffixes=("", "_media")
)

# drop duplicate join keys from gemius side if you do not need them
telep_merged_all = telep_merged_all.drop(columns=["county", "year"])

# clean up temporary names if you like
for var in ["map_place_size"]:
    if var in locals():
        del locals()[var]
del var

### Settlement - OEVK level connection

Read oevk-level electorate excel (modified sheet).


In [57]:
# read oevk-level electorate excel (modified sheet)
oevk_path = (
    data_dir.parent
    / "KSH"
    / "OevkLetszam.xlsx"
)

oevk_raw = pd.read_excel(
    oevk_path,
    sheet_name="modified"
)

# standardize column names to simple strings
oevk_raw.columns = oevk_raw.columns.astype(str).str.strip()

# rename hungarian columns to english
oevk = oevk_raw.rename(columns={
    "Vármegye/főváros": "county",
    "OEVK": "oevk_code",
    "Település": "settlement_name",
    "Választópolgárok": "electorate_oevk"
}).copy()

# helper: roman numeral to integer for budapest districts
roman_map = {
    "I": 1, "II": 2, "III": 3, "IV": 4, "V": 5,
    "VI": 6, "VII": 7, "VIII": 8, "IX": 9, "X": 10,
    "XI": 11, "XII": 12, "XIII": 13, "XIV": 14, "XV": 15,
    "XVI": 16, "XVII": 17, "XVIII": 18, "XIX": 19, "XX": 20,
    "XXI": 21, "XXII": 22, "XXIII": 23
}

def budapest_roman_to_arab(name):
    # work only on strings
    if not isinstance(name, str):
        return name

    text = name.strip()

    # only touch budapest entries
    if not text.startswith("Budapest"):
        return text

    # remove leading "Budapest"
    rest = text[len("Budapest"):].strip()

    # if there is extra text like "kerület", keep only the first token
    if " " in rest:
        first_token = rest.split()[0]
    else:
        first_token = rest

    # strip possible trailing dot
    roman = first_token.strip().strip(".")

    # if it is a known roman numeral, convert
    if roman in roman_map:
        num = roman_map[roman]
        return f"Budapest {num}. kerület"

    # otherwise leave as is
    return text

# apply conversion on settlement_name
oevk["settlement_name"] = oevk["settlement_name"].apply(budapest_roman_to_arab)

# prepare 2022 settlement-level electorate from telep_party_results_all
telep_2022 = (
    telep_party_results_all
    .loc[telep_party_results_all["year"] == 2022, ["settlement_name", "electorate_cand"]]
    .drop_duplicates()
)

# merge electorate_cand onto oevk table by settlement_name
oevk = oevk.merge(
    telep_2022,
    on="settlement_name",
    how="left"
)

# optional: clean up temporary variables
for var in [
    "oevk_path",
    "oevk_raw",
    "telep_2022",
    "roman_map",
    "budapest_roman_to_arab",
]:
    if var in locals():
        del locals()[var]
del var

#### weights

Weighting in cases where one settlement goes to multiple oevk.


In [58]:
# weighting in cases where one settlement goes to multiple oevk

# count how many rows each settlement has in the oevk table
oevk["settlement_row_count"] = (
    oevk
    .groupby("settlement_name")["oevk_code"]
    .transform("size")
)

# default weight is na
oevk["weight"] = pd.NA

# settlements that appear only once get weight 1
single_mask = oevk["settlement_row_count"] == 1
oevk.loc[single_mask, "weight"] = 1.0

# settlements that appear multiple times get electorate_oevk / electorate_cand
multi_mask = oevk["settlement_row_count"] > 1
oevk.loc[multi_mask, "weight"] = (
    oevk.loc[multi_mask, "electorate_oevk"]
    / oevk.loc[multi_mask, "electorate_cand"]
)

# oevk = oevk.drop(columns=["settlement_row_count"])

# clean up temporary variables
for var in ["single_mask", "multi_mask"]:
    if var in locals():
        del locals()[var]
del var

This block shows a quick diagnostic output so we can check the result.


In [59]:
# this block shows a quick diagnostic output so we can check the result.
print(oevk_party_results_all["county_name"].unique())

['BUDAPEST' 'BARANYA' 'BÁCS-KISKUN' 'BÉKÉS' 'BORSOD-ABAÚJ-ZEMPLÉN'
 'CSONGRÁD' 'FEJÉR' 'GYŐR-MOSON-SOPRON' 'HAJDÚ-BIHAR' 'HEVES'
 'JÁSZ-NAGYKUN-SZOLNOK' 'KOMÁROM-ESZTERGOM' 'NÓGRÁD' 'PEST' 'SOMOGY'
 'SZABOLCS-SZATMÁR-BEREG' 'TOLNA' 'VAS' 'VESZPRÉM' 'ZALA' nan]


This block shows a quick diagnostic output so we can check the result.


In [60]:
# this block shows a quick diagnostic output so we can check the result.
print(oevk["county"].unique())

['Szabolcs-Szatmár-Bereg' 'Borsod-Abaúj-Zemplén' 'Csongrád-Csanád'
 'Győr-Moson-Sopron' 'Vas' 'Somogy' 'Bács-Kiskun' 'Pest' 'Hajdú-Bihar'
 'Békés' 'Tolna' 'Baranya' 'Veszprém' 'Fejér' 'Zala' 'Heves'
 'Jász-Nagykun-Szolnok' 'Nógrád' 'Komárom-Esztergom' 'Budapest']


Build county name mapping: your 'county' -> oevk_party_results_all 'county_name'.


In [61]:
# build county name mapping: your 'county' -> oevk_party_results_all 'county_name'
county_name_map = {
    "Budapest": "BUDAPEST",
    "Baranya": "BARANYA",
    "Bács-Kiskun": "BÁCS-KISKUN",
    "Békés": "BÉKÉS",
    "Borsod-Abaúj-Zemplén": "BORSOD-ABAÚJ-ZEMPLÉN",
    "Csongrád-Csanád": "CSONGRÁD",  # old name in oevk_party_results_all
    "Fejér": "FEJÉR",
    "Győr-Moson-Sopron": "GYŐR-MOSON-SOPRON",
    "Hajdú-Bihar": "HAJDÚ-BIHAR",
    "Heves": "HEVES",
    "Jász-Nagykun-Szolnok": "JÁSZ-NAGYKUN-SZOLNOK",
    "Komárom-Esztergom": "KOMÁROM-ESZTERGOM",
    "Nógrád": "NÓGRÁD",
    "Pest": "PEST",
    "Somogy": "SOMOGY",
    "Szabolcs-Szatmár-Bereg": "SZABOLCS-SZATMÁR-BEREG",
    "Tolna": "TOLNA",
    "Vas": "VAS",
    "Veszprém": "VESZPRÉM",
    "Zala": "ZALA",
}

# make a small lookup table from oevk_party_results_all
county_lookup = (
    oevk_party_results_all[["county_name", "county_code"]]
    .drop_duplicates()
    .copy()
)

# map your county names to the uppercase county_name used in oevk_party_results_all
oevk["county_name_std"] = oevk["county"].map(county_name_map)

# merge county_code onto oevk
oevk = oevk.merge(
    county_lookup,
    left_on="county_name_std",
    right_on="county_name",
    how="left"
)

# optional: drop helper columns if you do not need them
oevk = oevk.drop(columns=["county_name_std", "county_name"])

# clean up helper objects
for var in ["county_name_map", "county_lookup"]:
    if var in locals():
        del locals()[var]
del var

Make a unique oevk id from county_code + oevk_code.


In [62]:
# make a unique oevk id from county_code + oevk_code
oevk["oevk_id"] = (
    oevk["county_code"].astype(str)
    + "_"
    + oevk["oevk_code"].astype(str)
)

# rename oevk table to oevk_weight
oevk_weight = oevk.copy()
del oevk

Adding year work on a copy of the weighted oevk table.


In [63]:
# adding year
# work on a copy of the weighted oevk table
base_oevk = oevk_weight.copy()

# list of election years you want
years = [2018, 2019, 2022, 2024]

# build one copy per year with a year column
oevk_list = []
for y in years:
    tmp = base_oevk.copy()
    tmp["year"] = y
    oevk_list.append(tmp)

# stack them on top of each other
oevk_weight = pd.concat(oevk_list, ignore_index=True)

# move year to the front
cols = ["year"] + [c for c in oevk_weight.columns if c != "year"]
oevk_weight = oevk_weight[cols]

# clean up temporary variables
for var in ["base_oevk", "years", "oevk_list", "tmp", "cols"]:
    if var in locals():
        del locals()[var]
del var, y


#### settlement results + oevk

Make a copy of oevk_weight as final_results.


In [64]:
# make a copy of oevk_weight as final_results
final_results = oevk_weight.copy()

# drop columns that are not needed in the final table
cols_to_drop = [
    "county_code",
    "electorate_cand",
    "electorate_oevk",
    "settlement_row_count",
]
final_results = final_results.drop(columns=cols_to_drop, errors="ignore")

# make sure year has the same type in both tables
final_results["year"] = pd.to_numeric(final_results["year"], errors="coerce")
telep_party_results_all["year"] = pd.to_numeric(
    telep_party_results_all["year"],
    errors="coerce"
)

# merge telep_party_results_all onto final_results by year and settlement_name
# if a settlement-year appears k times in oevk_weight and n times in telep_party_results_all,
# the result will contain k * n rows for that settlement-year combination
final_results = final_results.merge(
    telep_party_results_all,
    on=["year", "settlement_name"],
    how="left"
)

# clean up temporary variables
for var in ["cols_to_drop"]:
    if var in locals():
        del locals()[var]
del var

Define grouping keys: one row per year–oevk–party combo.


In [65]:
# define grouping keys: one row per year–oevk–party combo
group_cols = ["year", "oevk_id", "party_id"]

# numeric count columns to weight and sum
count_cols = [
    "electorate_cand",
    "voters_cand",
    "valid_cand_total",
    "electorate_list",
    "voters_list",
    "valid_list_total",
    "cand_votes",
    "list_votes",
]

# make a working copy
df = final_results.copy()

# remember original column order so we can restore it later
orig_cols = df.columns.tolist()

# make sure weight is numeric
df["weight"] = pd.to_numeric(df["weight"], errors="coerce")

# multiply count columns by weight so that groupby can just sum them
for col in count_cols:
    if col in df.columns:
        df[col] = df["weight"] * df[col]

# drop existing share columns; they will be recomputed after aggregation
for share_col in ["cand_share", "list_share"]:
    if share_col in df.columns:
        df = df.drop(columns=[share_col])

# build aggregation dictionary for all non-group columns
agg_dict = {}

for col in df.columns:
    if col in group_cols:
        # these are grouping keys, not aggregated
        continue
    elif col in count_cols:
        # weighted counts already in the column, just sum
        agg_dict[col] = "sum"
    elif col == "weight":
        # optionally keep sum of weights (could be useful for ellenőrzés)
        agg_dict[col] = "sum"
    else:
        # all other columns assumed constant within group -> take first
        agg_dict[col] = "first"

# group and aggregate
agg = (
    df
    .groupby(group_cols, as_index=False)
    .agg(agg_dict)
)

# recompute shares from weighted sums
agg["cand_share"] = (
    agg["cand_votes"]
    / agg["valid_cand_total"].replace({0: np.nan})
)

agg["list_share"] = (
    agg["list_votes"]
    / agg["valid_list_total"].replace({0: np.nan})
)

# restore column order as close to the original as possible
cols_final = [c for c in orig_cols if c in agg.columns] + [
    c for c in agg.columns if c not in orig_cols
]

final_results_oevk_party = agg[cols_final]

# recompute final oevk-level Others list votes as the residual after aggregation
# so district list shares cannot sum above 100 percent if one settlement source row is inconsistent
other_list_fix = (
    final_results_oevk_party[final_results_oevk_party["party_id"] != "OTHERS"]
    .groupby(["year", "oevk_id", "election_type"], as_index=False)
    .agg(
        non_other_list_votes=("list_votes", "sum"),
        valid_list_total=("valid_list_total", "first"),
    )
)
other_list_fix["recomputed_other_list_votes"] = (
    other_list_fix["valid_list_total"] - other_list_fix["non_other_list_votes"]
).clip(lower=0)
final_results_oevk_party = final_results_oevk_party.merge(
    other_list_fix[["year", "oevk_id", "election_type", "recomputed_other_list_votes"]],
    on=["year", "oevk_id", "election_type"],
    how="left",
)
other_mask = final_results_oevk_party["party_id"] == "OTHERS"
final_results_oevk_party.loc[other_mask, "list_votes"] = final_results_oevk_party.loc[
    other_mask, "recomputed_other_list_votes"
]
final_results_oevk_party["list_share"] = (
    final_results_oevk_party["list_votes"]
    / final_results_oevk_party["valid_list_total"].replace({0: np.nan})
)
final_results_oevk_party = final_results_oevk_party.drop(columns=["recomputed_other_list_votes"])

del other_list_fix, other_mask

# clean up temporary variables
for var in ["df", "agg", "agg_dict", "orig_cols", "cols_final", "count_cols", "group_cols"]:
    if var in locals():
        del locals()[var]
del var,col

final_results_oevk_party = final_results_oevk_party.drop(columns=["settlement_name"])

/var/folders/m_/_l_ts7n95x971syc73dnxsk00000gn/T/ipykernel_9260/939712438.py:62: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  / agg["valid_cand_total"].replace({0: np.nan})


This block prepares the next part of the notebook.


In [66]:
# this block prepares the next part of the notebook.
del final_results

#### settlement stats + oevk

Choose which year's rows to duplicate for 2026.


In [67]:
# choose which year's rows to duplicate for 2026
base_year_for_2026 = 2024

# select rows for the chosen base year
extra_2026 = oevk_weight[oevk_weight["year"] == base_year_for_2026].copy()

# change their year to 2026
extra_2026["year"] = 2026

# append these rows to the original table
oevk_weight = pd.concat([oevk_weight, extra_2026], ignore_index=True)

# optional check of row counts by year
print(oevk_weight["year"].value_counts().sort_index())

# clean up temporary variables
for var in ["extra_2026", "base_year_for_2026"]:
    if var in locals():
        del locals()[var]
del var

year
2018    3206
2019    3206
2022    3206
2024    3206
2026    3206
Name: count, dtype: int64


Helper to normalize budapest district names: "Budapest 03. kerület" -> "Budapest 3. kerület".


In [68]:
# helper to normalize budapest district names: "Budapest 03. kerület" -> "Budapest 3. kerület"
def normalize_budapest_name(name):
    if not isinstance(name, str):
        return name

    s = name.strip()

    # only touch strings starting with "Budapest"
    if not s.startswith("Budapest"):
        return s

    # match: "Budapest" + spaces + optional leading zeros + 1–2 digits + bármi utána
    # examples it should catch:
    # "Budapest 03. kerület"
    # "Budapest 3 kerület"
    # "Budapest 03.kerület"
    # "Budapest 03 ker."
    # "Budapest 03"
    m = re.match(r"^Budapest\s+0*(\d{1,2})\D.*$", s)
    if m:
        num = int(m.group(1))
        return f"Budapest {num}. kerület"

    # if no match, leave as is
    return s

# apply to settlement_name in telep_merged_all
telep_merged_all["settlement_name"] = (
    telep_merged_all["settlement_name"]
    .astype(str)
    .apply(normalize_budapest_name)
)

# optional: quick check of remaining weird forms
print(
    sorted(
        x for x in telep_merged_all["settlement_name"].unique()
        if x.startswith("Budapest 0")
    )[:20]
)

# optional: delete helper if you like
del normalize_budapest_name

[]


Make a copy of oevk_weight.


In [69]:
# make a copy of oevk_weight
final_statistics = oevk_weight.copy()

# drop columns that are not needed
cols_to_drop = [
    "county_code",
    "electorate_cand",
    "electorate_oevk",
    "settlement_row_count",
]
final_statistics = final_statistics.drop(columns=cols_to_drop, errors="ignore")

# make sure year types align (election year on the oevk side)
final_statistics["year"] = pd.to_numeric(final_statistics["year"], errors="coerce")

# prepare a merge-ready copy of telep_merged_all
telep_for_merge = telep_merged_all.copy()

# make sure target_election_year is numeric
telep_for_merge["target_election_year"] = pd.to_numeric(
    telep_for_merge["target_election_year"],
    errors="coerce"
)

# we do not need the original 'year' from ksh inside this merged table
telep_for_merge = telep_for_merge.drop(columns=["year"], errors="ignore")

# merge telep_merged_all onto final_statistics
# match by election year (year == target_election_year) and settlement_name
# if a settlement appears multiple times in oevk_weight,
# each row will get the same telep-level data (rows are duplicated as requested)
final_statistics = final_statistics.merge(
    telep_for_merge,
    left_on=["year", "settlement_name"],
    right_on=["target_election_year", "settlement_name"],
    how="left",
    suffixes=("", "_ksh")
)

# drop duplicate join key from the right side if you do not need it
final_statistics = final_statistics.drop(columns=["target_election_year"], errors="ignore")

# clean up temporary variables
for var in ["cols_to_drop", "telep_for_merge"]:
    if var in locals():
        del locals()[var]
del var

Grouping keys: one row per year–oevk combo.


In [70]:
# grouping keys: one row per year–oevk combo
group_cols = ["year", "oevk_id"]

# work on a copy
df = final_statistics.copy()

# make sure weight is numeric
df["weight"] = pd.to_numeric(df["weight"], errors="coerce")

# identify numeric columns
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

# remove group and weight cols from numeric list
for col in group_cols + ["weight"]:
    if col in numeric_cols:
        numeric_cols.remove(col)

# keep identifier-like numeric columns as metadata instead of aggregating them
for col in ["oevk_code", "year_media"]:
    if col in numeric_cols:
        numeric_cols.remove(col)

# meta columns (non-numeric or ids) that should stay constant within groups
meta_cols = [c for c in df.columns if c not in numeric_cols + ["weight"]]

# do not duplicate group_cols in meta_cols
meta_cols = [c for c in meta_cols if c not in group_cols]

# make a weighted copy of numeric columns
df_w = df.copy()
for col in numeric_cols:
    df_w[col] = df_w[col] * df_w["weight"]

# aggregate: sum weighted numerators and sum of weights
agg_num = (
    df_w
    .groupby(group_cols, as_index=False)
    .agg(
        {
            **{col: "sum" for col in numeric_cols},
            "weight": "sum"
        }
    )
)

# compute non-missing counts for each numeric column by group
counts = (
    df[group_cols + numeric_cols]
    .groupby(group_cols, as_index=False)
    .agg({col: "count" for col in numeric_cols})
)

# true settlement-level count variables should stay as weighted sums at OEVK level
weighted_sum_cols = [
    col for col in [
        "pop_total",
        "pop_0_14",
        "pop_65_plus",
        "pop_perm",
        "pop_perm_0_14_female",
        "pop_perm_65_plus_female",
        "pop_perm_0_14_male",
        "pop_perm_65_plus_male",
        "pop_perm_0_24",
        "budget_revenue",
        "budget_expenditure",
        "dwellings_built",
        "dwellings_terminated",
        "dwellings_built_muni",
        "dwellings_built_private",
        "gps_doctors",
        "pediatric_doctors",
        "cars",
        "internet_other",
        "tv_subs",
        "cars_first_reg",
        "post_offices",
        "internet_total",
        "internet_xdsl",
        "internet_cable",
        "internet_optical",
        "unemployed_total",
        "unemployed_180_plus",
        "unemployed_univ",
        "unemployed_bluecollar",
        "unemployed_young",
        "unemployed_365_plus",
    ]
    if col in numeric_cols
]

# convert non-additive weighted values to weighted means
for col in numeric_cols:
    if col not in weighted_sum_cols:
        agg_num[col] = agg_num[col] / agg_num["weight"].replace({0: np.nan})
    # if a group had only NA in this column, set result back to NA
    agg_num.loc[counts[col] == 0, col] = np.nan

# take meta information from original df (first row per group)
meta = (
    df[group_cols + meta_cols]
    .drop_duplicates(subset=group_cols, keep="first")
)

# merge meta with aggregated numeric columns
final_statistics_oevk = meta.merge(
    agg_num.drop(columns=["weight"]),
    on=group_cols,
    how="left"
)

# optional: reorder columns so ids come first
id_first = group_cols + [c for c in final_statistics_oevk.columns if c not in group_cols]
final_statistics_oevk = final_statistics_oevk[id_first]

# clean up temporary variables
for var in ["df", "df_w", "agg_num", "meta", "numeric_cols", "meta_cols", "group_cols", "id_first", "counts", "weighted_sum_cols"]:
    if var in locals():
        del locals()[var]
del var

### polls modification if nedded

### save csv-s

Output folder: data.parent/created.


In [71]:
# output folder: data.parent/created
created_dir = data_dir.parent / "created"
created_dir.mkdir(parents=True, exist_ok=True)

# 1) final_statistics_oevk → statistics.csv
final_statistics_oevk.to_csv(
    created_dir / "statistics.csv",
    index=False
)

# 2) final_results_oevk_party → oevk_results.csv
final_results_oevk_party.to_csv(
    created_dir / "oevk_results.csv",
    index=False
)

# 3) polls → polls.csv
polls.to_csv(
    created_dir / "polls.csv",
    index=False
)

# 4) mandates_all → global_results.csv
mandates_all_combined.to_csv(
    created_dir / "global_results.csv",
    index=False
)

# optional clean-up
for var in ["created_dir"]:
    if var in locals():
        del locals()[var]
del var